# DataMate — Làn D: chuẩn hoá + helper code-agent

Làn **1D** của planning DataMate. Kiến trúc `helper`: model viết code trên bảng đã làm sạch (cache profiler) + thư viện helper (kỳ thời gian, lọc đơn, xếp hạng, quy đổi).

**Cách dùng:** Add input dataset `thanhkhil/zaloai-datamate-dataset`, bật GPU T4 x2 + Internet, sửa `CONFIG` nếu cần rồi *Run All*.
- Mặc định chạy toàn bộ **public_test** và ghi `/kaggle/working/submission.jsonl` (nộp file này lên Public Leaderboard).
- Đo trên train: `split="train"`, `subset="dev_tune"` (hoặc `final_holdout`), sẽ có `merge_report.json` với PA theo shop / theo kiểu đáp án.
- Hết giờ giữa chừng: chạy lại version mới, attach output version cũ làm input và điền thư mục đó vào `resume_from_dirs` (cùng config).
- `finalize=True`: câu chưa xử lý xong được ghi `abstain` (0 điểm) để luôn có file nộp hợp lệ.

In [ ]:
CONFIG = {
  "track": "datamate",
  "experiment": "dm-1D-helper-qwen3-4b",
  "parent_run": None,
  "split": "public_test",
  "subset": "all",
  "manifest_seed": 2026,
  "limit": None,
  "mode": "helper",
  "input_roots": [
    "/kaggle/input"
  ],
  "out_dir": "/kaggle/working/dm_helper",
  "cache_dir": "/kaggle/working/dm_cache",
  "llm": {
    "backend": "auto",
    "model": "Qwen/Qwen3-4B-Instruct-2507",
    "revision": None,
    "dtype": "half",
    "max_model_len": 16384,
    "gpu_memory_utilization": 0.9,
    "enable_thinking": False,
    "load_in_4bit": False,
    "batch_size": 4,
    "seed": 0
  },
  "agent": {
    "max_new_tokens": 1536,
    "exec_timeout_s": 60,
    "exec_mem_mb": 6144,
    "exec_workers": 1,
    "repair": True,
    "rules_k": 8,
    "context_chars": 9000
  },
  "batch_size": 16,
  "cpu_threads": 1,
  "seed": 0,
  "wallclock_s": 37800.0,
  "finalize": True,
  "install_vllm": True,
  "resume_from_dirs": []
}

In [ ]:
import os, sys, json, time, glob, shutil, subprocess
from pathlib import Path
WORK = Path("/kaggle/working")
LIB = WORK / "lib"
(LIB / "dmlib").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB))
OUT = Path(CONFIG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-3000:]); return r.returncode

## Thư viện `dmlib` (nhúng sẵn)

In [ ]:
%%writefile /kaggle/working/lib/dmlib/__init__.py
"""DataMate library embedded into the Kaggle notebooks (see build_notebooks.py)."""

In [ ]:
%%writefile /kaggle/working/lib/dmlib/common.py
"""Run plumbing shared by the DataMate notebooks: durable JSONL shards with resume, stable sharding, config hashes,
atomic writes, input discovery under /kaggle/input and environment info."""
from __future__ import annotations

import hashlib
import json
import os
import platform
import subprocess
import time
from pathlib import Path


def dumps(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"), allow_nan=False, default=str)


def load_jsonl(path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def read_jsonl_tolerant(path) -> tuple[list[dict], int]:
    """Rows of a JSONL file written by a process that may have died mid-line. A broken *last* line is dropped
    (returns how many were dropped); a broken line in the middle means the file is corrupt -> ValueError."""
    rows, lines = [], Path(path).read_text(encoding="utf-8").split("\n")
    while lines and not lines[-1].strip():
        lines.pop()
    dropped = 0
    for i, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            if i == len(lines) - 1:
                dropped = 1
            else:
                raise ValueError(f"{path}: corrupt line {i + 1}")
    return rows, dropped


def atomic_write_text(path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_write_json(path, obj) -> None:
    atomic_write_text(path, json.dumps(obj, ensure_ascii=False, indent=1, default=str))


def sha256_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def config_hash(cfg: dict, ignore=()) -> str:
    """Hash of the settings that change results (shard/output/resume keys go in `ignore`)."""
    keep = {k: v for k, v in cfg.items() if k not in ignore}
    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]


def shard_of(key: str, num_shards: int) -> int:
    """Stable shard index (does not depend on PYTHONHASHSEED or on the order of the inputs)."""
    if num_shards <= 1:
        return 0
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % num_shards


class DurableJsonl:
    """Append-only result file of one worker. First line is a header with the run hash; every record is flushed and
    fsync'ed. On reopen: a truncated last line is dropped, the file is rewritten clean, and a header with another run
    hash is refused (never mix results of different configs). done_ids() = ids whose latest record is 'completed'."""

    def __init__(self, path, run_hash: str):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.run_hash = run_hash
        self.records: dict[str, dict] = {}
        if self.path.exists() and self.path.stat().st_size:
            rows, dropped = read_jsonl_tolerant(self.path)
            header = rows[0].get("_header") if rows else None
            if not header or header.get("run_hash") != run_hash:
                raise RuntimeError(f"{self.path} belongs to run {header and header.get('run_hash')}, not {run_hash}")
            for r in rows[1:]:
                self.records[r["id"]] = r
            if dropped:
                atomic_write_text(self.path, "\n".join(dumps(r) for r in rows) + "\n")
        else:
            atomic_write_text(self.path, dumps({"_header": {"run_hash": run_hash, "created": time.time()}}) + "\n")
        self._f = open(self.path, "a", encoding="utf-8")

    def done_ids(self) -> set[str]:
        return {i for i, r in self.records.items() if r.get("status") == "completed"}

    def append(self, rec: dict) -> None:
        self._f.write(dumps(rec) + "\n")
        self._f.flush()
        os.fsync(self._f.fileno())
        self.records[rec["id"]] = rec

    def close(self) -> None:
        if not self._f.closed:
            self._f.close()


def find_split_dir(split: str, roots=("/kaggle/input",), marker: str = "questions.jsonl", max_depth: int = 6) -> Path:
    """Directory `<...>/<split>` that holds `marker` (datasets get mounted as /kaggle/input/<slug>/<split>/<split>/...)."""
    for root in map(Path, roots):
        if not root.exists():
            continue
        base_depth = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            p = Path(dirpath)
            if len(p.parts) - base_depth > max_depth:
                dirnames[:] = []
                continue
            if p.name == split and marker in filenames:
                return p
    raise FileNotFoundError(f"no {split}/{marker} under {list(roots)}")


def env_info() -> dict:
    info = {"python": platform.python_version(), "platform": platform.platform(), "cpus": os.cpu_count()}
    try:
        info["gpus"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, timeout=20
        ).stdout.strip().splitlines()
    except Exception:
        info["gpus"] = []
    for mod in ("torch", "transformers", "vllm", "pandas"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            pass
    try:
        with open("/proc/meminfo") as f:
            info["mem_total_kb"] = int(f.readline().split()[1])
    except Exception:
        pass
    return info


def set_thread_env(threads: int = 1) -> None:
    """One BLAS/OpenMP thread per process (Kaggle has few CPUs)."""
    for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ.setdefault(k, str(threads))


class Deadline:
    """Wall-clock budget of a session: stop taking new work at `stop_after_s`."""

    def __init__(self, stop_after_s: float):
        self.t0 = time.time()
        self.stop_after_s = stop_after_s

    def elapsed(self) -> float:
        return time.time() - self.t0

    def expired(self) -> bool:
        return self.elapsed() >= self.stop_after_s

In [ ]:
%%writefile /kaggle/working/lib/dmlib/parsers.py
"""Parse the messy cells of shop files: VN money/number strings, dates, quantities with units, order status."""
from __future__ import annotations

import datetime as dt
import functools
import math
import re
import unicodedata

STATUSES = ("hoàn thành", "huỷ", "trả hàng", "đang giao")


def fold(s) -> str:
    """Lower-case, strip Vietnamese diacritics (đ -> d), collapse spaces."""
    if s is None:
        return ""
    return _fold_str(str(s))


@functools.lru_cache(maxsize=200_000)
def _fold_str(s: str) -> str:
    t = unicodedata.normalize("NFD", s.lower()).replace("đ", "d")
    return " ".join("".join(c for c in t if not unicodedata.combining(c)).split())


def _is_missing(v) -> bool:
    return v is None or (isinstance(v, float) and math.isnan(v))


def _plain_number(s: str):
    """'227,262,100' / '1.000.000' / '1,090,000.0000' / '16 076' / '1,45' / '0.4' -> float, or None."""
    s = s.strip().replace(" ", " ")
    if not s:
        return None
    sign = -1 if s.startswith("-") else 1
    s = s.lstrip("+-").strip()
    if re.fullmatch(r"\d{1,3}( \d{3})+", s):
        return sign * float(s.replace(" ", ""))
    if not re.fullmatch(r"[\d.,]+", s) or not re.search(r"\d", s):
        return None
    dots, commas = s.count("."), s.count(",")
    if dots and commas:
        # the separator that comes last is the decimal one
        dec = "." if s.rfind(".") > s.rfind(",") else ","
        thou = "," if dec == "." else "."
        s = s.replace(thou, "").replace(dec, ".")
    elif dots > 1 or commas > 1:
        sep = "." if dots else ","
        if not all(len(g) == 3 for g in s.split(sep)[1:]):
            return None
        s = s.replace(sep, "")
    elif dots == 1 or commas == 1:
        sep = "." if dots else ","
        head, tail = s.split(sep)
        # one group of exactly 3 digits after the separator is a thousands separator (VN money style)
        s = head + tail if len(tail) == 3 and head and head != "0" else head + "." + tail
    try:
        return sign * float(s)
    except ValueError:
        return None


def _as_int_if_whole(x: float):
    return int(round(x)) if abs(x - round(x)) < 1e-9 else x


def parse_number(v):
    """Number cell -> int/float or None. Handles '144.000đ', '750k', '12tr460', '1,45tr', '2 triệu'."""
    if _is_missing(v) or isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return _as_int_if_whole(float(v))
    s = fold(v).replace("vnd", "").replace("dong", "").strip()
    s = re.sub(r"(?<=\d)\s*d$", "", s).strip()
    if not s:
        return None
    m = re.fullmatch(r"(-?[\d.,]+)\s*(tr|trieu)\s*(\d{1,3})?", s)
    if m:
        base = _plain_number(m.group(1))
        if base is None:
            return None
        extra = int(m.group(3).ljust(3, "0")) * 1000 if m.group(3) else 0
        return _as_int_if_whole(base * 1_000_000 + (extra if base >= 0 else -extra))
    m = re.fullmatch(r"(-?[\d.,]+)\s*(k|nghin|ngan)", s)
    if m:
        base = _plain_number(m.group(1))
        return None if base is None else _as_int_if_whole(base * 1000)
    x = _plain_number(s)
    return None if x is None else _as_int_if_whole(x)


_THG = re.compile(r"(\d{1,2})\s*thg\s*(\d{1,2})\s*(\d{4})")
_DMY = re.compile(r"(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})")
_YMD = re.compile(r"(\d{4})-(\d{1,2})-(\d{1,2})")
_DM = re.compile(r"(\d{1,2})[/\-.](\d{1,2})")


def _safe_date(y, m, d):
    try:
        return dt.date(int(y), int(m), int(d))
    except ValueError:
        return None


def parse_date(v, default_year: int | None = None):
    """Date cell -> datetime.date or None. A 'd/m' cell without a year takes `default_year` (the sheet's year)."""
    if _is_missing(v):
        return None
    if isinstance(v, dt.datetime):
        return v.date()
    if isinstance(v, dt.date):
        return v
    if hasattr(v, "to_pydatetime"):  # pandas Timestamp
        return v.to_pydatetime().date()
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        # Excel serial day number (1899-12-30 epoch); only plausible years 1990..2060
        if 32874 <= v <= 58440:
            return dt.date(1899, 12, 30) + dt.timedelta(days=int(v))
        return None
    s = str(v).strip()
    for rx, order in ((_YMD, "ymd"), (_THG, "dmy"), (_DMY, "dmy")):
        m = rx.search(s)
        if m:
            a, b, c = m.groups()
            return _safe_date(a, b, c) if order == "ymd" else _safe_date(c, b, a)
    m = _DM.fullmatch(s)
    if m and default_year:
        return _safe_date(default_year, m.group(2), m.group(1))
    return None


def parse_qty(v):
    """'3 m' -> (3, 'm'); '1,5 kg' -> (1.5, 'kg'); 2 -> (2, None)."""
    if _is_missing(v):
        return (None, None)
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return (_as_int_if_whole(float(v)), None)
    m = re.fullmatch(r"\s*(-?[\d.,]+)\s*(.*?)\s*", str(v))
    if not m:
        return (None, None)
    num = m.group(1)
    if "," in num and "." not in num and len(num.split(",")[1]) != 3:
        num = num.replace(",", ".")
    q = parse_number(num)
    return (q, m.group(2) or None) if q is not None else (None, None)


_STATUS_KEYS = [
    ("huỷ", ("huy", "cancel", "cancelled", "canceled", "bom hang", "khach bom")),
    ("trả hàng", ("tra hang", "tra lai", "hoan hang", "return", "doi tra", "tra mot phan", "tra 1 phan", "returned", "refund", "refunded")),
    ("đang giao", ("dang giao", "dang ship", "shipping", "dang van chuyen", "cho giao", "dang chuyen", "delivering", "in transit", "shipped")),
    ("hoàn thành", ("hoan thanh", "da xong", "xong", "done", "complete", "completed", "thanh cong", "da giao", "ht", "ok", "da thanh toan")),
]


_STATUS_RX = [
    (status, re.compile("(^|[^a-z])(" + "|".join(re.escape(k) for k in keys) + ")([^a-z]|$)")) for status, keys in _STATUS_KEYS
]


def norm_status(v):
    """Map a status / note cell to one of the four rulebook statuses, or None if it names none."""
    f = fold(v)
    if not f:
        return None
    for status, rx in _STATUS_RX:
        if rx.search(f):
            return status
    return None

In [ ]:
%%writefile /kaggle/working/lib/dmlib/periods.py
"""Rulebook §2 time periods. Every period is an inclusive (start, end) pair of datetime.date."""
from __future__ import annotations

import calendar as _cal
import datetime as dt

import pandas as pd

Period = tuple  # (dt.date, dt.date), both inclusive


def to_date(d) -> dt.date:
    if isinstance(d, dt.datetime):
        return d.date()
    if isinstance(d, dt.date):
        return d
    return pd.Timestamp(d).date()


def _month_end(y: int, m: int) -> dt.date:
    return dt.date(y, m, _cal.monthrange(y, m)[1])


def day(d) -> Period:
    d = to_date(d)
    return (d, d)


def yesterday(as_of) -> Period:
    return day(to_date(as_of) - dt.timedelta(days=1))


def this_week(as_of) -> Period:
    """Monday of the week holding as_of .. as_of."""
    d = to_date(as_of)
    return (d - dt.timedelta(days=d.weekday()), d)


def last_week(as_of) -> Period:
    """The whole previous Monday..Sunday week."""
    monday = this_week(as_of)[0]
    return (monday - dt.timedelta(days=7), monday - dt.timedelta(days=1))


def last_n_days(as_of, n: int) -> Period:
    """N consecutive days ending today (today included)."""
    d = to_date(as_of)
    return (d - dt.timedelta(days=n - 1), d)


def this_month(as_of) -> Period:
    d = to_date(as_of)
    return (d.replace(day=1), d)


def same_days_prev_month(as_of) -> Period:
    """Day 1 .. same day number of the previous month (or that month's end if it has no such day)."""
    d = to_date(as_of)
    y, m = (d.year, d.month - 1) if d.month > 1 else (d.year - 1, 12)
    end = min(d.day, _cal.monthrange(y, m)[1])
    return (dt.date(y, m, 1), dt.date(y, m, end))


def prev_month_full(as_of) -> Period:
    d = to_date(as_of)
    y, m = (d.year, d.month - 1) if d.month > 1 else (d.year - 1, 12)
    return (dt.date(y, m, 1), _month_end(y, m))


def month(m: int, year: int | None = None, as_of=None) -> Period:
    """Whole month M. Without a year: this year unless that month starts after as_of, then last year."""
    if year is None:
        d = to_date(as_of)
        year = d.year if dt.date(d.year, m, 1) <= d else d.year - 1
    return (dt.date(year, m, 1), _month_end(year, m))


def quarter(q: int, year: int | None = None, as_of=None) -> Period:
    """Calendar quarter Q (Q1 = Jan..Mar). Without a year: same roll-back rule as month()."""
    first = 3 * (q - 1) + 1
    if year is None:
        d = to_date(as_of)
        year = d.year if dt.date(d.year, first, 1) <= d else d.year - 1
    return (dt.date(year, first, 1), _month_end(year, first + 2))


def year_to_date(as_of) -> Period:
    d = to_date(as_of)
    return (dt.date(d.year, 1, 1), d)


def whole_year(year: int) -> Period:
    return (dt.date(year, 1, 1), dt.date(year, 12, 31))


def prev_period(p: Period) -> Period:
    """The period of the same length right before p."""
    start, end = to_date(p[0]), to_date(p[1])
    n = (end - start).days + 1
    return (start - dt.timedelta(days=n), start - dt.timedelta(days=1))


def clip_to_asof(p: Period, as_of) -> Period:
    """Records after today have not happened yet."""
    return (to_date(p[0]), min(to_date(p[1]), to_date(as_of)))


def _lunar_to_solar(calendar: pd.DataFrame, lday: int, lmonth: int, lyear: int) -> dt.date:
    cal = calendar
    leap_col = "lunar_leap" if "lunar_leap" in cal.columns else ("is_leap_month" if "is_leap_month" in cal.columns else None)
    mask = (cal["lunar_day"].astype(int) == lday) & (cal["lunar_month"].astype(int) == lmonth) & (cal["lunar_year"].astype(int) == lyear)
    if leap_col is not None:
        mask &= ~cal[leap_col].astype(str).str.lower().isin(["true", "1"])
    hit = cal.loc[mask, "date"]
    if hit.empty:
        raise KeyError(f"lunar {lday}/{lmonth}/{lyear} not in calendar.csv")
    return to_date(sorted(hit.astype(str))[0])


def tet_range(year: int, calendar: pd.DataFrame) -> Period:
    """Tết of year Y: solar date of 23 tháng Chạp (lunar year Y-1) .. solar date of mùng 10 tháng Giêng (lunar year Y)."""
    return (_lunar_to_solar(calendar, 23, 12, year - 1), _lunar_to_solar(calendar, 10, 1, year))


def before_tet_range(year: int, calendar: pd.DataFrame) -> Period:
    """Same number of days right before 23 tháng Chạp."""
    return prev_period(tet_range(year, calendar))


def shift_of(ts) -> str | None:
    """Ca sáng 06:00–13:59, ca chiều 14:00–21:59 (by order creation time); other hours: None."""
    if ts is None or pd.isna(ts):
        return None
    h = pd.Timestamp(ts).hour
    if 6 <= h <= 13:
        return "sáng"
    if 14 <= h <= 21:
        return "chiều"
    return None


def count_weekday(p: Period, weekday: int) -> int:
    """Number of days with that weekday (Monday=0) in p, days without orders included."""
    days = pd.date_range(to_date(p[0]), to_date(p[1]), freq="D")
    return int((days.day_of_week == weekday).sum())


def in_range(dates: pd.Series, p: Period) -> pd.Series:
    """Boolean mask: date (or datetime) within the inclusive period."""
    d = pd.to_datetime(dates, errors="coerce").dt.normalize()
    return (d >= pd.Timestamp(to_date(p[0]))) & (d <= pd.Timestamp(to_date(p[1])))


def overlap_status(p: Period, data_range: Period) -> str:
    """'outside' (no day of p has data -> unanswerable), 'partial' (compute on the covered part) or 'inside'."""
    s, e = to_date(p[0]), to_date(p[1])
    ds, de = to_date(data_range[0]), to_date(data_range[1])
    if e < ds or s > de:
        return "outside"
    if s < ds or e > de:
        return "partial"
    return "inside"


def pct_change(cur, prev, ndigits: int = 2):
    """Percent change vs the previous period (negative when it fell); None when prev is 0."""
    if prev in (0, None) or cur is None:
        return None
    return round((cur - prev) / prev * 100, ndigits)

In [ ]:
%%writefile /kaggle/working/lib/dmlib/answer.py
"""Answer formatting (rulebook §9: one format per type) and the local scorer used on train/dev.

Scorer assumptions (not confirmed against the organisers' scorer): the predicted `type` must equal the gold type;
numbers match within 1e-6; percent is compared after rounding both sides to 2 decimals; strings match the gold answer or
any alias after norm_string; table columns are matched by normalised name, else by position.
"""
from __future__ import annotations

import math
import re
import unicodedata
from decimal import ROUND_HALF_UP, Decimal

from .parsers import parse_number

TYPES = ("number", "percent", "string", "date", "boolean", "list", "table", "unanswerable", "abstain")


class AnswerError(ValueError):
    """The value cannot be written in the single format its type allows."""


def round_half_up(x: float, ndigits: int = 0):
    q = Decimal(1).scaleb(-ndigits)
    r = Decimal(repr(float(x))).quantize(q, rounding=ROUND_HALF_UP)
    return int(r) if ndigits == 0 else float(r)


def norm_string(s) -> str:
    """NFC, lower-case, every char that is not a letter/digit/space -> space, collapse spaces. Diacritics kept."""
    t = unicodedata.normalize("NFC", str(s)).lower()
    t = "".join(c if (c.isalnum() or c.isspace()) else " " for c in t)
    return " ".join(t.split())


def _finite_number(v):
    if isinstance(v, bool):
        raise AnswerError("boolean is not a number")
    if hasattr(v, "item") and not isinstance(v, (list, dict, str)):  # numpy scalar
        v = v.item()
    if isinstance(v, str):
        p = parse_number(v)
        if p is None:
            raise AnswerError(f"not a number: {v!r}")
        v = p
    if not isinstance(v, (int, float)) or isinstance(v, bool):
        raise AnswerError(f"not a number: {v!r}")
    if isinstance(v, float) and not math.isfinite(v):
        raise AnswerError("NaN/Infinity is not valid JSON")
    return v


def _fmt_number(v, money: bool = False):
    v = _finite_number(v)
    if money:
        return round_half_up(v, 0)
    if isinstance(v, int):
        return v
    r = round_half_up(v, 2)
    return int(r) if r == int(r) else r


def _fmt_scalar_cell(v):
    if isinstance(v, str):
        return v
    return _fmt_number(v)


_DATE = re.compile(r"\d{4}-\d{2}(-\d{2})?")


def format_value(typ: str, value, money: bool = False):
    if typ == "number":
        return _fmt_number(value, money)
    if typ == "percent":
        return _fmt_number(round_half_up(_finite_number(value), 2))
    if typ == "string":
        if not isinstance(value, str) or not value.strip():
            raise AnswerError("string answer must be a non-empty string")
        return unicodedata.normalize("NFC", value.strip())
    if typ == "date":
        if hasattr(value, "strftime"):
            value = value.strftime("%Y-%m-%d")
        if not isinstance(value, str) or not _DATE.fullmatch(value):
            raise AnswerError(f"date must be YYYY-MM-DD or YYYY-MM: {value!r}")
        return value
    if typ == "boolean":
        if hasattr(value, "item") and not isinstance(value, (bool, str)):
            value = value.item()
        if not isinstance(value, bool):
            raise AnswerError("boolean must be true/false")
        return value
    if typ == "list":
        if isinstance(value, tuple):
            value = list(value)
        if not isinstance(value, list):
            raise AnswerError("list answer must be a JSON array")
        return [_fmt_scalar_cell(x) for x in value]
    if typ == "table":
        if not isinstance(value, dict) or not isinstance(value.get("columns"), list) or not isinstance(value.get("rows"), list):
            raise AnswerError("table must be {'columns': [...], 'rows': [[...]]}")
        cols = [str(c) for c in value["columns"]]
        rows = []
        for r in value["rows"]:
            if not isinstance(r, (list, tuple)) or len(r) != len(cols):
                raise AnswerError("every table row must have one cell per column")
            rows.append([_fmt_scalar_cell(x) for x in r])
        return {"columns": cols, "rows": rows}
    raise AnswerError(f"unknown type {typ!r}")


def format_answer(qid: str, typ: str, value, reason: str | None = None, money: bool = False) -> dict:
    """Submission line for one question. Raises AnswerError if the value does not fit the type's format."""
    if typ == "abstain":
        return {"id": qid, "type": "abstain"}
    if typ == "unanswerable":
        return {"id": qid, "type": "unanswerable", "reason": str(reason or "không có dữ liệu")}
    if typ not in TYPES:
        raise AnswerError(f"unknown type {typ!r}")
    return {"id": qid, "type": typ, "answer": format_value(typ, value, money)}


def validate_record(rec: dict) -> str | None:
    """None if the submission line is well formed, else the reason."""
    if not isinstance(rec, dict) or not isinstance(rec.get("id"), str):
        return "missing id"
    typ = rec.get("type")
    if typ == "abstain":
        return None
    if typ == "unanswerable":
        return None if isinstance(rec.get("reason", ""), str) else "reason must be a string"
    if "answer" not in rec:
        return "missing answer"
    try:
        out = format_value(typ, rec["answer"])
    except AnswerError as e:
        return str(e)
    if typ in ("number", "percent") and not isinstance(rec["answer"], (int, float)):
        return "number must be a JSON number"
    return None if out is not None else "bad answer"


# ---------------------------------------------------------------- scoring
def _num_eq(p, g, percent=False) -> bool:
    if isinstance(p, bool) or not isinstance(p, (int, float)) or isinstance(g, bool) or not isinstance(g, (int, float)):
        return False
    if percent:
        return abs(round_half_up(p, 2) - round_half_up(g, 2)) < 1e-6
    return abs(p - g) < 1e-6


def _cell_eq(p, g, aliases=()) -> bool:
    if isinstance(g, str):
        return isinstance(p, str) and norm_string(p) in {norm_string(x) for x in (g, *aliases)}
    return _num_eq(p, g)


def _table_eq(p, g) -> bool:
    if not isinstance(p, dict) or not isinstance(p.get("columns"), list) or not isinstance(p.get("rows"), list):
        return False
    gcols, pcols = [norm_string(c) for c in g["columns"]], [norm_string(c) for c in p["columns"]]
    if len(gcols) != len(pcols) or len(g["rows"]) != len(p["rows"]):
        return False
    idx = [pcols.index(c) if c in pcols else i for i, c in enumerate(gcols)]
    if sorted(idx) != list(range(len(gcols))):
        idx = list(range(len(gcols)))
    try:
        prows = [[r[i] for i in idx] for r in p["rows"]]
    except (IndexError, TypeError):
        return False
    unused = list(range(len(prows)))
    for grow in g["rows"]:
        hit = next((j for j in unused if all(_cell_eq(pc, gc) for pc, gc in zip(prows[j], grow))), None)
        if hit is None:
            return False
        unused.remove(hit)
    return True


def score(pred: dict | None, gold: dict) -> int:
    """+1 correct, 0 abstain / missing / unreadable, -1 wrong."""
    if not pred or pred.get("type") in (None, "abstain"):
        return 0
    ptype, gtype = pred.get("type"), gold["type"]
    if ptype == "unanswerable":
        if not isinstance(pred.get("reason", ""), str):
            return 0
        return 1 if gtype == "unanswerable" else -1
    if ptype != gtype or "answer" not in pred:
        return -1
    p, g, aliases = pred["answer"], gold.get("answer"), gold.get("alias") or []
    if gtype == "number":
        ok = _num_eq(p, g)
    elif gtype == "percent":
        ok = _num_eq(p, g, percent=True)
    elif gtype == "string":
        ok = _cell_eq(p, g, aliases)
    elif gtype == "date":
        ok = isinstance(p, str) and p == g
    elif gtype == "boolean":
        ok = isinstance(p, bool) and p == g
    elif gtype == "list":
        ok = isinstance(p, list) and len(p) == len(g) and all(_cell_eq(a, b) for a, b in zip(p, g))
    elif gtype == "table":
        ok = _table_eq(p, g)
    else:
        ok = False
    return 1 if ok else -1


def penalized_accuracy(preds: dict, golds: dict) -> dict:
    """preds/golds keyed by question id; ids missing from preds count as abstain."""
    s = {qid: score(preds.get(qid), g) for qid, g in golds.items()}
    n = len(s)
    correct = sum(v == 1 for v in s.values())
    wrong = sum(v == -1 for v in s.values())
    return {
        "n": n,
        "correct": correct,
        "wrong": wrong,
        "abstain": n - correct - wrong,
        "pa": (correct - wrong) / n if n else 0.0,
        "accuracy_answered": correct / (correct + wrong) if correct + wrong else 0.0,
        "coverage": (correct + wrong) / n if n else 0.0,
        "per_id": s,
    }

In [ ]:
%%writefile /kaggle/working/lib/dmlib/loader.py
"""Read every table of a shop (xlsx sheets, csv, sqlite) as raw cells, with the header row detected and the source
of every row kept in _file / _sheet / _row. No cleaning here: see profiler.py."""
from __future__ import annotations

import csv
import io
import re
import sqlite3
import warnings
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

from .parsers import parse_date, parse_number

LOADER_VERSION = "1"
LINEAGE = ("_file", "_sheet", "_row")


@dataclass
class RawTable:
    name: str
    file: str
    sheet: str | None
    header_row: int | None  # 0-based row of the header inside the sheet / file
    df: pd.DataFrame


def _is_labelish(v) -> bool:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return False
    if not isinstance(v, str) or not v.strip():
        return False
    return parse_number(v) is None and parse_date(v) is None


def detect_header_row(raw: pd.DataFrame, scan: int = 15) -> int:
    """Row with the most label-like cells among the first `scan` rows; ties go to the later row only when the earlier
    row is a sparse group header (e.g. 'Đơn | Hàng | Tiền' above the real column names)."""
    best, best_score = 0, -1
    for i in range(min(scan, len(raw))):
        score = sum(_is_labelish(v) for v in raw.iloc[i].tolist())
        if score > best_score:
            best, best_score = i, score
    return best


def _unique_names(cells) -> list[str]:
    names, seen = [], {}
    for j, c in enumerate(cells):
        n = str(c).strip() if _is_labelish(c) or (isinstance(c, str) and c.strip()) else f"col_{j}"
        n = re.sub(r"\s+", " ", n)
        if n in seen:
            seen[n] += 1
            n = f"{n}_{seen[n]}"
        else:
            seen[n] = 1
        names.append(n)
    return names


def frame_from_grid(raw: pd.DataFrame, file: str, sheet: str | None, header_row: int | None = None) -> tuple[pd.DataFrame, int]:
    if header_row is None:
        header_row = detect_header_row(raw)
    body = raw.iloc[header_row + 1 :].copy()
    body.columns = _unique_names(raw.iloc[header_row].tolist())
    body = body.dropna(how="all")
    body = body.loc[:, [c for c in body.columns if not (c.startswith("col_") and body[c].isna().all())]]
    body.insert(len(body.columns), "_file", file)
    body.insert(len(body.columns), "_sheet", sheet or "")
    body.insert(len(body.columns), "_row", body.index.to_numpy() + 1)  # 1-based row number in the source grid
    return body.reset_index(drop=True), header_row


def _slug(s: str) -> str:
    return re.sub(r"[^\w\-]+", "_", s, flags=re.UNICODE).strip("_")


def read_csv_grid(path: Path) -> pd.DataFrame:
    raw = path.read_bytes()
    for enc in ("utf-8-sig", "cp1258", "latin-1"):
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    try:
        delim = csv.Sniffer().sniff(text[:5000], delimiters=",;\t|").delimiter
    except csv.Error:
        delim = ","
    return pd.read_csv(io.StringIO(text), header=None, dtype=str, sep=delim, keep_default_na=False, na_values=[""])


def read_excel_grids(path: Path) -> dict[str, pd.DataFrame]:
    """All sheets as raw grids. calamine is ~10x faster than openpyxl; fall back when it is not installed."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            return pd.read_excel(path, sheet_name=None, header=None, engine="calamine")
        except ImportError:
            return pd.read_excel(path, sheet_name=None, header=None)


def load_shop(shop_dir) -> list[RawTable]:
    shop_dir = Path(shop_dir)
    out: list[RawTable] = []
    for path in sorted(shop_dir.iterdir()):
        suf = path.suffix.lower()
        if suf in (".xlsx", ".xlsm", ".xls"):
            sheets = read_excel_grids(path)
            for sheet, grid in sheets.items():
                if grid.dropna(how="all").empty:
                    continue
                df, h = frame_from_grid(grid, path.name, sheet)
                name = _slug(path.stem) if len(sheets) == 1 and sheet.lower().startswith("sheet") else f"{_slug(path.stem)}__{_slug(sheet)}"
                out.append(RawTable(name, path.name, sheet, h, df))
        elif suf in (".csv", ".tsv", ".txt"):
            grid = read_csv_grid(path)
            if grid.empty:
                continue
            df, h = frame_from_grid(grid, path.name, None)
            out.append(RawTable(_slug(path.stem), path.name, None, h, df))
        elif suf in (".sqlite", ".db", ".sqlite3"):
            con = sqlite3.connect(f"file:{path}?mode=ro", uri=True)
            try:
                for (t,) in con.execute("select name from sqlite_master where type='table'"):
                    df = pd.read_sql_query(f'select * from "{t}"', con)
                    df["_file"], df["_sheet"], df["_row"] = path.name, t, range(1, len(df) + 1)
                    out.append(RawTable(f"{_slug(path.stem)}__{_slug(t)}", path.name, t, None, df))
            finally:
                con.close()
    return out

In [ ]:
%%writefile /kaggle/working/lib/dmlib/profiler.py
"""Lane 0 (CPU): profile every shop once and cache clean tables for the helper code-agent.

For each table: typed columns (VN money/number strings, dates, quantities with units), order-level cells forward-filled
inside an order (rulebook §8), exact duplicate lines of the same order dropped, `_status` normalised to the four
statuses. Raw cells are cached next to the clean ones (lineage kept). Plus README facts (data cut-off, unit
conversions) and a capability profile with three states: present / absent_per_readme / unknown.
Nothing here is a semantic mapping to orders/order_lines yet: that is decided after the first round's error analysis.
"""
from __future__ import annotations

import datetime as dt
import json
import re
import shutil
from pathlib import Path

import pandas as pd

from .common import atomic_write_json, atomic_write_text, sha256_file
from .loader import LINEAGE, LOADER_VERSION, RawTable, load_shop
from .parsers import fold, norm_status, parse_date, parse_number, parse_qty

PROFILER_VERSION = "2"

_ID_HINTS = ("ma ", "ma_", "barcode", "sdt", "dien thoai", "so phieu", "bien so", "so dt", "phone", " id", "code", "imei", "serial")
_ORDER_KEY_HINTS = ("ma don", "so don", "ma hd", "so hd", "ma hoa don", "so hoa don", "ma phieu", "so phieu", "order", "invoice", "don hang", "ma bill", "so bill", "bill")
_CAPABILITIES = {
    "status": ("trang thai", "status", "tinh trang"),
    "customer": ("khach", "customer"),
    "customer_group": ("nhom khach", "loai khach", "hang khach", "phan khuc"),
    "staff": ("nhan vien", "thu ngan", "nguoi ban", "tho ", "staff", "cashier", "nguoi lam"),
    "purchase": ("nhap", "nha cung cap", "ncc", "supplier"),
    "stock": ("ton", "stock", "inventory"),
    "cost": ("gia von", "cost", "gia nhap"),
    "payment_method": ("thanh toan", "hinh thuc", "payment", "pttt", "phuong thuc"),
    "debt": ("no ", "cong no", "con no", "ghi no", "debt", "tra gop", "han no"),
    "expiry": ("han dung", "han su dung", "hsd", "expiry", "het han"),
    "time_of_day": ("gio", "time", "thoi gian"),
    "shipping_fee": ("phi ship", "phi giao", "van chuyen", "shipping"),
    "discount": ("giam gia", "chiet khau", "voucher", "khuyen mai", "discount"),
    "returns": ("tra hang", "tra lai", "hoan"),
}


def parse_readme(text: str) -> dict:
    """Data cut-off date and unit conversions ('X: 1 thùng = 24 lon') written in a shop README."""
    info: dict = {"data_until": None, "conversions": {}}
    m = re.search(r"đến hết (?:ngày )?(\d{1,2})/(\d{1,2})/(\d{4})", text)
    if m:
        info["data_until"] = dt.date(int(m.group(3)), int(m.group(2)), int(m.group(1))).isoformat()
    for line in text.splitlines():
        m = re.match(r"\s*[-*]\s*(.+?):\s*1\s+(.+?)\s*=\s*([\d.,]+)\s*(.+?)\s*$", line)
        if not m:
            continue
        item, big, n, base = m.group(1).strip(), m.group(2).strip(), parse_number(m.group(3)), m.group(4).strip()
        if n is None:
            continue
        conv = info["conversions"].setdefault(item, {})
        conv[big] = n
        conv["_base"] = base
    return info


def _year_hint(*names) -> int | None:
    for n in names:
        m = re.search(r"(20\d{2})", str(n or ""))
        if m:
            return int(m.group(1))
    return None


def map_unique(values: pd.Series, fn) -> pd.Series:
    """values.map(fn), computing fn once per distinct value (shop files repeat the same cells a lot)."""
    keys = values.map(lambda v: (type(v).__name__, v) if not (isinstance(v, float) and pd.isna(v)) else None)
    cache: dict = {}
    out = []
    for k, v in zip(keys, values):
        if k not in cache:
            cache[k] = fn(v)
        out.append(cache[k])
    return pd.Series(out, index=values.index, dtype=object)


def _infer_column(name: str, values: pd.Series, year: int | None) -> dict:
    nn = values.dropna()
    nn = nn[nn.astype(str).str.strip() != ""]
    info = {"kind": "empty" if nn.empty else "text", "non_null": int(len(nn)), "nulls": int(len(values) - len(nn))}
    if nn.empty:
        return info
    fname = " " + fold(name) + " "
    sample = nn if len(nn) <= 3000 else nn.sample(3000, random_state=0)
    idlike = any(h in fname for h in _ID_HINTS)
    n_ok = map_unique(sample, lambda v: parse_number(v) is not None).astype(bool).mean()
    d_ok = map_unique(sample, lambda v: parse_date(v, year) is not None).astype(bool).mean()
    is_dt = sample.map(lambda v: isinstance(v, (pd.Timestamp, dt.datetime)) and (v.hour or v.minute or v.second)).mean()
    q = map_unique(sample, parse_qty)
    q_ok = q.map(lambda x: x[0] is not None).mean()
    q_unit = q.map(lambda x: x[1] is not None).mean()
    if d_ok >= 0.9 and not (n_ok >= 0.9 and sample.map(lambda v: isinstance(v, (int, float))).mean() > 0.5):
        info["kind"] = "datetime" if is_dt >= 0.5 else "date"
    elif n_ok >= 0.9 and not idlike:
        info["kind"] = "number"
    elif q_ok >= 0.9 and q_unit >= 0.2 and not idlike:
        info["kind"] = "qty"
    if info["kind"] == "text":
        vc = nn.astype(str).value_counts()
        info["n_unique"] = int(len(vc))
        info["top"] = {str(k): int(v) for k, v in vc.head(12).items()}
    return info


def _clean_column(values: pd.Series, kind: str, year: int | None):
    if kind == "number":
        return pd.to_numeric(map_unique(values, parse_number), errors="coerce"), None
    if kind == "date":
        return pd.to_datetime(map_unique(values, lambda v: parse_date(v, year)), errors="coerce"), None
    if kind == "datetime":
        def conv(v):
            if isinstance(v, (pd.Timestamp, dt.datetime)):
                return pd.Timestamp(v)
            ts = pd.to_datetime(v, errors="coerce", dayfirst=True)
            if pd.isna(ts):
                d = parse_date(v, year)
                return pd.Timestamp(d) if d else pd.NaT
            return ts
        return pd.to_datetime(map_unique(values, conv), errors="coerce"), None
    if kind == "qty":
        parsed = map_unique(values, parse_qty)
        return pd.to_numeric(parsed.map(lambda x: x[0]), errors="coerce"), parsed.map(lambda x: x[1])
    return values.map(lambda v: None if v is None or (isinstance(v, float) and pd.isna(v)) else str(v).strip()), None


_ORDER_KEY_EXACT = {"hd", "ma", "so ct", "ct", "so chung tu", "ma ct", "so", "ma don", "don"}


def _order_key(df: pd.DataFrame, cols: dict) -> str | None:
    for c in df.columns:
        if c in LINEAGE or cols.get(c, {}).get("kind") in ("number", "date", "datetime", "empty"):
            continue
        f = fold(c)
        if f in _ORDER_KEY_EXACT or any(h in " " + f + " " for h in _ORDER_KEY_HINTS):
            return c
    return None


_SUMMARY_ROW = re.compile(r"^(tong cong|tong so|tong|cong)\b")


def _drop_summary_rows(df: pd.DataFrame, key: str | None) -> tuple[pd.DataFrame, int]:
    """Footer lines such as 'Tổng cộng' (no order key, a text cell starting with tổng/cộng)."""
    data_cols = [c for c in df.columns if c not in LINEAGE]
    def is_summary(row) -> bool:
        if key is not None and pd.notna(row.get(key)):
            return False
        return any(isinstance(v, str) and _SUMMARY_ROW.match(fold(v)) for v in row[data_cols].tolist())
    mask = df.apply(is_summary, axis=1) if len(df) else pd.Series([], dtype=bool)
    return df.loc[~mask], int(mask.sum())


def _ffill_order_level(df: pd.DataFrame, key: str, min_groups: int = 1) -> tuple[pd.DataFrame, list[str]]:
    """Orders are runs of lines with the same key (a blank key continues the order above). Forward-fill the key and
    every column that is constant inside multi-line orders and blank on some later line whose order's first line has
    it (rulebook §8). Needs `min_groups` multi-line orders to decide, so one stray line cannot trigger a fill."""
    data_cols = [c for c in df.columns if c not in LINEAGE]
    filled_key = df[key].ffill()
    grp = (filled_key != filled_key.shift()).cumsum()
    size = grp.map(grp.value_counts())
    multi = size > 1
    n_multi = int(grp[multi].nunique())
    out = df.copy()
    filled = []
    other = [c for c in data_cols if c != key]
    if df[key].isna().any() and df.loc[df[key].isna() & filled_key.notna(), other].notna().any(axis=None):
        out[key] = filled_key
        filled.append(key)
    if n_multi < min_groups:
        return out, filled
    is_first = ~grp.duplicated()
    for c in other:
        s = df[c]
        sub = pd.DataFrame({"g": grp[multi], "v": s[multi]}).dropna()
        if sub.empty:
            continue
        nuniq = sub.groupby("g")["v"].nunique()
        if (nuniq <= 1).mean() < 0.98:
            continue
        has_first = s.where(is_first).groupby(grp).transform("first").notna()
        gaps = s.isna() & ~is_first & has_first & multi
        if gaps.any():
            out[c] = s.groupby(grp).ffill()
            filled.append(c)
    return out, [c for c in df.columns if c in filled]


def clean_table(t: RawTable, readme_text: str) -> tuple[pd.DataFrame, dict]:
    df = t.df
    year = _year_hint(t.sheet, t.file)
    cols = {c: _infer_column(c, df[c], year) for c in df.columns if c not in LINEAGE}
    meta: dict = {"file": t.file, "sheet": t.sheet, "header_row": t.header_row, "rows_raw": int(len(df)), "year_hint": year}
    key = _order_key(df, cols)
    meta["order_key"] = key
    df, meta["summary_rows_dropped"] = _drop_summary_rows(df, key)
    work = df
    ffilled: list[str] = []
    if key is not None:
        work, ffilled = _ffill_order_level(df, key)
    meta["ffilled_columns"] = ffilled
    clean = pd.DataFrame(index=work.index)
    for c in work.columns:
        if c in LINEAGE:
            clean[c] = work[c]
            continue
        vals, unit = _clean_column(work[c], cols[c]["kind"], year)
        clean[c] = vals
        if unit is not None:
            clean[f"{c}__unit"] = unit
    dropped = 0
    if key is not None:
        data_cols = [c for c in clean.columns if c not in LINEAGE]
        dup = clean.duplicated(subset=data_cols, keep="first")
        dropped = int(dup.sum())
        clean = clean.loc[~dup]
    meta["duplicate_rows_dropped"] = dropped
    status_col = next((c for c in cols if "trang thai" in fold(c) or fold(c) in ("status", "tinh trang")), None)
    note_col = next((c for c in cols if "ghi chu" in fold(c) or fold(c) == "note"), None)
    if status_col is not None:
        clean["_status"] = map_unique(clean[status_col], norm_status)
        meta["status_from"] = status_col
    elif note_col is not None and key is not None:
        # rulebook §8: no status column -> status is in Ghi chú; a note naming no status is a completed order
        clean["_status"] = map_unique(clean[note_col], lambda v: norm_status(v) or "hoàn thành")
        meta["status_from"] = f"{note_col} (no status column)"
    if "_status" in clean:
        meta["status_counts"] = {str(k): int(v) for k, v in clean["_status"].value_counts(dropna=False).items()}
        unknown = clean.loc[clean["_status"].isna(), meta["status_from"].split(" (")[0]].dropna().astype(str).value_counts()
        meta["status_unmapped"] = {str(k): int(v) for k, v in unknown.head(10).items()}
    for c, info in cols.items():
        if info["kind"] in ("number",):
            s = clean[c]
            info.update(min=None if s.isna().all() else float(s.min()), max=None if s.isna().all() else float(s.max()))
        if info["kind"] in ("date", "datetime"):
            s = clean[c]
            info.update(min=None if s.isna().all() else str(s.min().date()), max=None if s.isna().all() else str(s.max().date()))
            info["parse_fail"] = int(s.isna().sum() - info["nulls"])
        examples = work[c].dropna().astype(str).head(3).tolist()
        info["examples"] = examples
    meta["columns"] = cols
    meta["rows_clean"] = int(len(clean))
    return clean.reset_index(drop=True), meta


def capability_profile(tables_meta: dict, readme_text: str) -> dict:
    all_cols = " | ".join(" " + fold(c) + " " for m in tables_meta.values() for c in m["columns"])
    readme = " " + fold(readme_text) + " "
    out = {}
    for cap, hints in _CAPABILITIES.items():
        if any(h in all_cols for h in hints):
            out[cap] = "present"
        elif "## cot" in readme or "cot trong" in readme:
            # the README lists the columns of every file and none matches: absent per README, still not proven
            out[cap] = "unknown" if any(h in readme for h in hints) else "absent_per_readme"
        else:
            out[cap] = "unknown"
    return out


_NOT_ACTIVITY_DATE = ("sinh", "han", "hsd", "expiry", "dang ky", "phai tra", "het han", "bao hanh")


def _date_range(tables_meta: dict) -> tuple[str | None, str | None]:
    """Span of activity dates: date columns of order tables (fallback: every table), minus birthdays, expiry and
    due dates, which lie far outside the data period."""
    lo, hi = None, None
    metas = [m for m in tables_meta.values() if m.get("order_key")] or list(tables_meta.values())
    for m in metas:
        for c, info in m["columns"].items():
            if any(re.search(rf"\b{h}\b", fold(c)) for h in _NOT_ACTIVITY_DATE):
                continue
            if info["kind"] in ("date", "datetime") and info.get("min"):
                lo = info["min"] if lo is None or info["min"] < lo else lo
                hi = info["max"] if hi is None or info["max"] > hi else hi
    return lo, hi


def shop_card(prof: dict, readme_text: str, max_cols: int = 40) -> str:
    """Compact text the helper code-agent reads: README + every clean table with typed columns and examples."""
    lines = [f"# Shop {prof['shop']}", "", readme_text.strip(), "", "## Bảng đã làm sạch (T['<tên>'] là pandas DataFrame)"]
    lines.append(f"Khoảng ngày thấy trong file: {prof['date_min']} .. {prof['date_max']}; README: dữ liệu đến hết {prof['data_until']}")
    for name, m in prof["tables"].items():
        src = m["file"] + (f" / sheet {m['sheet']}" if m["sheet"] else "")
        lines.append(f"\n### T['{name}']  ({src}; {m['rows_clean']} dòng)")
        notes = []
        if m.get("order_key"):
            notes.append(f"khoá đơn: {m['order_key']}")
        if m.get("ffilled_columns"):
            notes.append("đã ffill trong đơn: " + ", ".join(m["ffilled_columns"]))
        if m.get("duplicate_rows_dropped"):
            notes.append(f"đã bỏ {m['duplicate_rows_dropped']} dòng trùng cùng đơn")
        if m.get("status_from"):
            notes.append(f"_status (4 trạng thái chuẩn) lấy từ {m['status_from']}: {m.get('status_counts')}")
        if notes:
            lines.append("- " + "; ".join(notes))
        for i, (c, info) in enumerate(m["columns"].items()):
            if i >= max_cols:
                lines.append(f"- … {len(m['columns']) - max_cols} cột nữa")
                break
            extra = ""
            if info["kind"] in ("number", "date", "datetime") and info.get("min") is not None:
                extra = f" [{info['min']} .. {info['max']}]"
            elif info["kind"] == "text" and info.get("n_unique", 99) <= 12:
                extra = " giá trị: " + ", ".join(list(info.get("top", {}))[:12])
            elif info["kind"] == "qty":
                extra = f" (số; đơn vị ở cột '{c}__unit')"
            ex = "; ".join(info.get("examples", [])[:3])
            lines.append(f"- `{c}` {info['kind']}{extra} — vd: {ex}")
    lines.append("\n## Năng lực dữ liệu: " + ", ".join(f"{k}={v}" for k, v in prof["capabilities"].items()))
    return "\n".join(lines)


def cache_key(shop_dir: Path, extra: str = "") -> str:
    import hashlib

    h = hashlib.sha256(f"{LOADER_VERSION}/{PROFILER_VERSION}/{extra}".encode())
    for p in sorted(shop_dir.iterdir()):
        if p.is_file():
            h.update(p.name.encode())
            h.update(sha256_file(p).encode())
    return h.hexdigest()[:20]


def _to_parquet(df: pd.DataFrame, path: Path) -> None:
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].map(lambda v: None if v is None or (isinstance(v, float) and pd.isna(v)) else str(v))
    df.to_parquet(path, index=False)


def profile_shop(shop_dir, out_root, split: str, rulebook_sha: str = "", force: bool = False) -> dict:
    """Profile one shop into out_root/<shop>/ (tables/*.parquet, raw/*.parquet, profile.json, card.md, COMPLETE)."""
    shop_dir = Path(shop_dir)
    out = Path(out_root) / shop_dir.name
    key = cache_key(shop_dir, f"{split}/{rulebook_sha}")
    marker = out / "COMPLETE"
    if not force and marker.exists() and marker.read_text().strip() == key:
        prof = json.loads((out / "profile.json").read_text(encoding="utf-8"))
        prof["reused"] = True
        return prof
    tmp = out.with_name(out.name + ".partial")
    shutil.rmtree(tmp, ignore_errors=True)
    (tmp / "tables").mkdir(parents=True)
    (tmp / "raw").mkdir()
    readme_path = shop_dir / "README.md"
    readme = readme_path.read_text(encoding="utf-8") if readme_path.exists() else ""
    tables_meta, errors = {}, {}
    for t in load_shop(shop_dir):
        try:
            clean, meta = clean_table(t, readme)
            _to_parquet(clean, tmp / "tables" / f"{t.name}.parquet")
            _to_parquet(t.df.astype(object), tmp / "raw" / f"{t.name}.parquet")
            tables_meta[t.name] = meta
        except Exception as e:  # one bad table must not hide the others
            errors[t.name] = f"{type(e).__name__}: {e}"
    rinfo = parse_readme(readme)
    lo, hi = _date_range(tables_meta)
    prof = {
        "shop": shop_dir.name,
        "split": split,
        "cache_key": key,
        "loader_version": LOADER_VERSION,
        "profiler_version": PROFILER_VERSION,
        "data_until": rinfo["data_until"],
        "conversions": rinfo["conversions"],
        "date_min": lo,
        "date_max": hi,
        "tables": tables_meta,
        "errors": errors,
        "capabilities": capability_profile(tables_meta, readme),
        "files": sorted(p.name for p in shop_dir.iterdir() if p.is_file()),
    }
    atomic_write_json(tmp / "profile.json", prof)
    atomic_write_text(tmp / "card.md", shop_card(prof, readme))
    atomic_write_text(tmp / "COMPLETE", key)
    shutil.rmtree(out, ignore_errors=True)
    tmp.rename(out)
    return prof


def load_cached_shop(shop_cache_dir) -> tuple[dict[str, pd.DataFrame], dict]:
    d = Path(shop_cache_dir)
    if not (d / "COMPLETE").exists():
        raise FileNotFoundError(f"{d} has no COMPLETE marker")
    prof = json.loads((d / "profile.json").read_text(encoding="utf-8"))
    tables = {p.stem: pd.read_parquet(p) for p in sorted((d / "tables").glob("*.parquet"))}
    return tables, prof

In [ ]:
%%writefile /kaggle/working/lib/dmlib/rulebook.py
"""Split the split's rulebook.md into rules and pick the ones a question needs.

The rulebook changes between splits (new industries, new rules), so it is read at inference time. The general
sections (time, unanswerable, file conventions, answer format, words, revenue basics) always go into the prompt;
ranking / debt / stock rules are retrieved by word overlap with the question; an industry subsection is included
when the shop README mentions that industry.
"""
from __future__ import annotations

import math
import re
from collections import Counter
from dataclasses import dataclass

from .parsers import fold

ALWAYS = ("don hang va doanh thu", "thoi gian", "khong tra loi duoc", "cach ghi trong file", "dap an", "tu ngu")
INDUSTRY_ALIASES = {
    "ca phe": ("ca phe", "cafe", "coffee", "tra sua", "do uong"),
    "nha thuoc": ("nha thuoc", "thuoc", "duoc"),
    "vat lieu xay dung": ("vat lieu xay dung", "vlxd", "xi mang", "gach", "sat thep"),
    "me va be": ("me va be", "bim", "ta em be", "sua bot"),
    "sua xe": ("sua xe", "tiem xe", "xe may", "bien so"),
    "cua hang dien thoai": ("dien thoai", "iphone", "thu may cu", "tra gop"),
}


@dataclass
class Rule:
    section: str
    subsection: str | None
    title: str
    text: str  # the markdown row (or paragraph)
    header: str  # table header of the row, kept so the rule reads on its own


def _tokens(s: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", fold(s))


class Rulebook:
    def __init__(self, text: str):
        self.text = text
        self.preamble = ""
        self.sections: dict[str, str] = {}  # section title -> full markdown
        self.rules: list[Rule] = []
        self._parse()
        df = Counter(t for r in self.rules for t in set(_tokens(r.title + " " + r.text)))
        n = max(len(self.rules), 1)
        self.idf = {t: math.log(1 + n / c) for t, c in df.items()}

    def _parse(self) -> None:
        section = sub = None
        header = ""
        pre: list[str] = []
        buf: dict[str, list[str]] = {}
        lines = self.text.splitlines()
        for i, line in enumerate(lines):
            if line.startswith("## "):
                section, sub, header = line[3:].strip(), None, ""
                buf[section] = [line]
                continue
            if section is None:
                if not line.startswith("# "):
                    pre.append(line)
                continue
            buf[section].append(line)
            if line.startswith("### "):
                sub, header = line[4:].strip(), ""
                continue
            if line.startswith("|"):
                cells = [c.strip() for c in line.strip().strip("|").split("|")]
                nxt = lines[i + 1] if i + 1 < len(lines) else ""
                if re.fullmatch(r"\|?\s*-+.*", nxt) and set(nxt.replace("|", "").strip()) <= set("-: "):
                    header = line + "\n" + nxt
                    continue
                if set(line.replace("|", "").strip()) <= set("-: "):
                    continue
                self.rules.append(Rule(section, sub, re.sub(r"\*", "", cells[0]), line, header))
            elif line.strip().startswith("- "):
                self.rules.append(Rule(section, sub, line.strip()[2:60], line, ""))
        self.preamble = "\n".join(pre).strip()
        self.sections = {k: "\n".join(v).strip() for k, v in buf.items()}

    @staticmethod
    def _is_always(section: str) -> bool:
        f = fold(section)
        return any(a in f for a in ALWAYS)

    def _industries(self, industry_text: str) -> set[str]:
        f = " " + fold(industry_text) + " "
        out = set()
        for r in self.rules:
            if r.subsection:
                key = fold(r.subsection)
                aliases = INDUSTRY_ALIASES.get(key, (key,))
                if any(a in f for a in aliases):
                    out.add(r.subsection)
        return out

    def score(self, rule: Rule, question: str) -> float:
        q = set(_tokens(question))
        title = set(_tokens(rule.title))
        body = set(_tokens(rule.text))
        return sum(3 * self.idf.get(t, 0) for t in q & title) + sum(self.idf.get(t, 0) for t in q & (body - title))

    def select(self, question: str, industry_text: str = "", k: int = 8) -> str:
        """Markdown with the always-on sections, the shop's industry rules and the top-k other rules."""
        parts = [self.preamble] if self.preamble else []
        parts += [txt for title, txt in self.sections.items() if self._is_always(title)]
        industries = self._industries(industry_text)
        picked: dict[tuple, list[Rule]] = {}
        for r in self.rules:
            if self._is_always(r.section):
                continue
            if r.subsection and r.subsection in industries:
                picked.setdefault((r.section, r.subsection, r.header), []).append(r)
        others = [r for r in self.rules if not self._is_always(r.section) and not r.subsection]
        ranked = sorted(((self.score(r, question), j, r) for j, r in enumerate(others)), key=lambda x: (-x[0], x[1]))
        for s, _, r in ranked[:k]:
            if s > 0:
                picked.setdefault((r.section, None, r.header), []).append(r)
        for (section, sub, header), rules in picked.items():
            head = f"## {section}" + (f"\n### {sub}" if sub else "")
            parts.append(head + ("\n" + header if header else "") + "\n" + "\n".join(r.text for r in rules))
        return "\n\n".join(p for p in parts if p)

In [ ]:
%%writefile /kaggle/working/lib/dmlib/helpers.py
"""Stable arithmetic the helper code-agent calls (lane D). Business policy (what counts as revenue, debt, ties) stays
in the split's rulebook; these functions only implement the defaults that rulebook §1/§3/§9/§10 spell out."""
from __future__ import annotations

import pandas as pd

from .answer import round_half_up
from .parsers import fold
from .periods import in_range, pct_change  # noqa: F401  (re-exported for the agent)

_RANK = {"huỷ": 3, "trả hàng": 2, "đang giao": 1, "hoàn thành": 0}


def order_status(df: pd.DataFrame, order_col: str, status_col: str = "_status") -> pd.Series:
    """One status per order: a cancelled or (even partly) returned line marks the whole order."""
    st = df[status_col].fillna("hoàn thành").map(lambda s: _RANK.get(s, 0))
    worst = st.groupby(df[order_col]).max()
    inv = {v: k for k, v in _RANK.items()}
    return worst.map(inv)


def revenue_lines(df: pd.DataFrame, order_col: str, status_col: str = "_status") -> pd.DataFrame:
    """Lines of orders that count for revenue / order counts: not cancelled, not returned (whole order), đang giao kept."""
    st = order_status(df, order_col, status_col)
    ok = set(st[st.isin(["hoàn thành", "đang giao"])].index)
    return df[df[order_col].isin(ok)]


def rank(df: pd.DataFrame, value_col: str, name_col: str, code_col: str | None = None, k: int | None = None, ascending: bool = False) -> pd.DataFrame:
    """Sort by value (descending by default); equal values go by code ascending, else by name ascending."""
    tie = code_col or name_col
    out = df.sort_values([value_col, tie], ascending=[ascending, True], kind="mergesort")
    return out.head(k) if k else out


def base_factor(item: str, unit: str | None, conversions: dict) -> float:
    """How many base units one `unit` of `item` is (README / conversion table). Unknown unit or item -> 1."""
    if unit is None or (isinstance(unit, float) and pd.isna(unit)):
        return 1
    fi, fu = fold(item), fold(unit)
    for name, conv in conversions.items():
        if fold(name) != fi:
            continue
        if fold(conv.get("_base", "")) == fu:
            return 1
        for big, n in conv.items():
            if big != "_base" and fold(big) == fu:
                return n
    return 1


def money(x) -> int:
    """VND amount rounded to the đồng, half up."""
    return round_half_up(float(x), 0)


def pct(part, whole, ndigits: int = 2):
    """part / whole in percent, 2 decimals; None when whole is 0."""
    if not whole:
        return None
    return round_half_up(part / whole * 100, ndigits)

In [ ]:
%%writefile /kaggle/working/lib/dmlib/executor.py
"""Run model-written code in a separate Python process with a timeout, memory/CPU limits and capped output.

The code ends by calling submit(type, value) (or unanswerable(reason) / abstain()); submit() formats the answer with
dmlib.answer, so a format error surfaces as an execution error the agent can repair. Labels never enter this process:
the code only gets the shop folder, calendar and cache paths of its context. A static deny-list blocks label files,
network, shells and file deletion; this is a guard rail, not a filesystem sandbox.
"""
from __future__ import annotations

import json
import os
import re
import resource
import subprocess
import sys
import tempfile
import time
from dataclasses import dataclass, field
from pathlib import Path

LIB_ROOT = str(Path(__file__).resolve().parent.parent)  # folder that holds the dmlib package

_BLOCKED = [
    (r"answers?\.jsonl|answers?\b['\"]", "label file"),
    (r"\b(import|from)\s+(requests|urllib|http|socket|aiohttp|httpx|ftplib|smtplib)\b", "network"),
    (r"\b(import|from)\s+subprocess\b|os\.system|os\.popen|os\.exec|pty\.spawn", "shell"),
    (r"shutil\.rmtree|os\.remove|os\.unlink|os\.rmdir|\.unlink\(", "file deletion"),
    (r"/kaggle/input|\.\./", "path outside the sandbox"),
]

_RUNNER = r'''
import json, sys, traceback
sys.path.insert(0, {lib_root!r})
import dmlib.answer as _A
_CTX = json.load(open({ctx_path!r}, encoding="utf-8"))
globals().update(_CTX)
_OUT = {result_path!r}
_DONE = []

def _write(rec):
    _DONE.append(rec)
    with open(_OUT, "w", encoding="utf-8") as f:
        json.dump({{"ok": True, "record": rec}}, f, ensure_ascii=False, allow_nan=False)

def submit(type, value=None, reason=None, money=False):
    """Final answer: submit('number', 48250000), submit('list', ['a', 'b']), submit('percent', -8.25) ..."""
    _write(_A.format_answer(QID, type, value, reason=reason, money=money))

def unanswerable(reason="dữ liệu không đủ để trả lời"):
    _write(_A.format_answer(QID, "unanswerable", None, reason=str(reason)))

def abstain(reason=None):
    _write({{"id": QID, "type": "abstain"}})

try:
{preload}
    _code = open({code_path!r}, encoding="utf-8").read()
    exec(compile(_code, "<agent_code>", "exec"), globals())
    if not _DONE:
        raise RuntimeError("code finished without calling submit(), unanswerable() or abstain()")
except BaseException as e:
    tb = traceback.format_exc()
    with open(_OUT, "w", encoding="utf-8") as f:
        json.dump({{"ok": False, "error": tb[-4000:]}}, f, ensure_ascii=False)
'''


@dataclass
class ExecResult:
    ok: bool
    record: dict | None = None
    error: str | None = None
    stdout: str = ""
    seconds: float = 0.0
    extra: dict = field(default_factory=dict)


def static_check(code: str) -> str | None:
    """Reason the code is refused, or None."""
    for rx, why in _BLOCKED:
        if re.search(rx, code):
            return f"blocked: {why} ({rx})"
    return None


def _limits(mem_mb: int, cpu_s: int):
    def apply():
        os.setsid()
        resource.setrlimit(resource.RLIMIT_AS, (mem_mb << 20, mem_mb << 20))
        resource.setrlimit(resource.RLIMIT_CPU, (cpu_s, cpu_s + 5))
    return apply


def run_code(
    code: str,
    qid: str,
    context: dict,
    timeout_s: float = 60,
    mem_mb: int = 6144,
    max_output_chars: int = 4000,
    preload: str = "",
    workdir: str | None = None,
) -> ExecResult:
    """Execute `code` with `context` variables (JSON values) plus QID; `preload` runs first (e.g. load cached tables)."""
    why = static_check(code)
    if why:
        return ExecResult(False, error=why)
    t0 = time.time()
    with tempfile.TemporaryDirectory(prefix="dm_exec_", dir=workdir) as tmp:
        tmp = Path(tmp)
        ctx_path, code_path, result_path = tmp / "ctx.json", tmp / "code.py", tmp / "result.json"
        ctx_path.write_text(json.dumps({**context, "QID": qid}, ensure_ascii=False, default=str), encoding="utf-8")
        code_path.write_text(code, encoding="utf-8")
        pre = "\n".join("    " + l for l in (preload or "pass").splitlines())
        runner = _RUNNER.format(lib_root=LIB_ROOT, ctx_path=str(ctx_path), result_path=str(result_path), code_path=str(code_path), preload=pre)
        (tmp / "runner.py").write_text(runner, encoding="utf-8")
        env = {
            "PATH": os.environ.get("PATH", "/usr/bin:/bin"),
            "HOME": str(tmp),
            "PYTHONDONTWRITEBYTECODE": "1",
            "PYTHONIOENCODING": "utf-8",
            "OMP_NUM_THREADS": "1",
            "MKL_NUM_THREADS": "1",
            "OPENBLAS_NUM_THREADS": "1",
            "MPLBACKEND": "Agg",
        }
        try:
            proc = subprocess.run(
                [sys.executable, str(tmp / "runner.py")],
                cwd=tmp,
                env=env,
                capture_output=True,
                text=True,
                timeout=timeout_s,
                preexec_fn=_limits(mem_mb, int(timeout_s) + 5),
            )
            out = (proc.stdout or "") + (("\n[stderr]\n" + proc.stderr) if proc.stderr else "")
        except subprocess.TimeoutExpired as e:
            out = (e.stdout or b"").decode("utf-8", "replace") if isinstance(e.stdout, bytes) else (e.stdout or "")
            return ExecResult(False, error=f"timeout after {timeout_s}s", stdout=out[:max_output_chars], seconds=time.time() - t0)
        stdout = out if len(out) <= max_output_chars else out[: max_output_chars // 2] + "\n…[cắt bớt]…\n" + out[-max_output_chars // 2 :]
        if not result_path.exists():
            return ExecResult(False, error=f"process exited with code {proc.returncode} without a result", stdout=stdout, seconds=time.time() - t0)
        res = json.loads(result_path.read_text(encoding="utf-8"))
        if res.get("ok"):
            return ExecResult(True, record=res["record"], stdout=stdout, seconds=time.time() - t0)
        return ExecResult(False, error=res.get("error"), stdout=stdout, seconds=time.time() - t0)

In [ ]:
%%writefile /kaggle/working/lib/dmlib/llm.py
"""Greedy chat generation behind one interface: vLLM (fast, Kaggle T4 in fp16), HF transformers (fallback) or a mock.

Prompts are rendered with the model's own chat template (enable_thinking passed when the template knows it) and
prompts that do not fit `max_model_len - max_tokens` are not generated but returned with finish='too_long'.
"""
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from typing import Callable


@dataclass
class GenOut:
    text: str
    prompt_tokens: int
    output_tokens: int
    finish: str  # stop | length | too_long | error


def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


class BaseLLM:
    max_model_len: int = 16384

    def render(self, conv: list[dict]) -> str:
        raise NotImplementedError

    def count_tokens(self, conv: list[dict]) -> int:
        raise NotImplementedError

    def generate(self, convs: list[list[dict]], max_tokens: int) -> list[GenOut]:
        raise NotImplementedError


def _apply_template(tok, conv, thinking: bool) -> str:
    try:
        return tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
    except TypeError:
        return tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)


class VLLM(BaseLLM):
    def __init__(self, model: str, revision: str | None = None, dtype: str = "half", max_model_len: int = 16384,
                 gpu_memory_utilization: float = 0.90, quantization: str | None = None, seed: int = 0,
                 enable_thinking: bool = False, tensor_parallel_size: int = 1, enforce_eager: bool = False, **_):
        from vllm import LLM, SamplingParams  # noqa: F401

        self._SamplingParams = SamplingParams
        self.llm = LLM(model=model, revision=revision, dtype=dtype, max_model_len=max_model_len, seed=seed,
                       gpu_memory_utilization=gpu_memory_utilization, quantization=quantization,
                       tensor_parallel_size=tensor_parallel_size, enable_prefix_caching=True, enforce_eager=enforce_eager)
        self.tok = self.llm.get_tokenizer()
        self.max_model_len = max_model_len
        self.thinking = enable_thinking

    def render(self, conv):
        return _apply_template(self.tok, conv, self.thinking)

    def count_tokens(self, conv):
        return len(self.tok(self.render(conv), add_special_tokens=False)["input_ids"])

    def generate(self, convs, max_tokens):
        prompts = [self.render(c) for c in convs]
        lens = [len(self.tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
        todo = [i for i, n in enumerate(lens) if n + max_tokens <= self.max_model_len]
        outs = [GenOut("", n, 0, "too_long") for n in lens]
        if todo:
            sp = self._SamplingParams(temperature=0.0, top_p=1.0, max_tokens=max_tokens, seed=0)
            res = self.llm.generate([prompts[i] for i in todo], sp, use_tqdm=False)
            for i, r in zip(todo, res):
                o = r.outputs[0]
                outs[i] = GenOut(o.text, lens[i], len(o.token_ids), "length" if o.finish_reason == "length" else "stop")
        return outs


class HFLLM(BaseLLM):
    def __init__(self, model: str, revision: str | None = None, dtype: str = "half", max_model_len: int = 16384,
                 load_in_4bit: bool = False, enable_thinking: bool = False, batch_size: int = 4, device_map="auto", **_):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(model, revision=revision)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        kw = {"revision": revision, "device_map": device_map}
        if load_in_4bit:
            from transformers import BitsAndBytesConfig

            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
        else:
            kw["torch_dtype"] = torch.float16 if dtype in ("half", "float16") else (torch.bfloat16 if dtype == "bfloat16" else torch.float32)
        self.model = AutoModelForCausalLM.from_pretrained(model, **kw).eval()
        self.max_model_len = max_model_len
        self.thinking = enable_thinking
        self.batch_size = batch_size

    def render(self, conv):
        return _apply_template(self.tok, conv, self.thinking)

    def count_tokens(self, conv):
        return len(self.tok(self.render(conv), add_special_tokens=False)["input_ids"])

    def generate(self, convs, max_tokens):
        prompts = [self.render(c) for c in convs]
        lens = [len(self.tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
        outs = [GenOut("", n, 0, "too_long") for n in lens]
        todo = [i for i, n in enumerate(lens) if n + max_tokens <= self.max_model_len]
        todo.sort(key=lambda i: lens[i])
        for b in range(0, len(todo), self.batch_size):
            idx = todo[b : b + self.batch_size]
            enc = self.tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)
            with self.torch.no_grad():
                gen = self.model.generate(**enc, max_new_tokens=max_tokens, do_sample=False, temperature=None, top_p=None,
                                          top_k=None, pad_token_id=self.tok.pad_token_id)
            for row, i in zip(gen, idx):
                new = row[enc["input_ids"].shape[1] :]
                n_out = int((new != self.tok.pad_token_id).sum())
                text = self.tok.decode(new, skip_special_tokens=True)
                outs[i] = GenOut(text, lens[i], n_out, "length" if n_out >= max_tokens else "stop")
        return outs


class MockLLM(BaseLLM):
    """Deterministic stand-in for tests and CPU dry-runs: fn(conv) -> text."""

    def __init__(self, fn: Callable[[list[dict]], str], max_model_len: int = 16384, **_):
        self.fn = fn
        self.max_model_len = max_model_len

    def render(self, conv):
        return "\n".join(f"<{m['role']}>{m['content']}" for m in conv)

    def count_tokens(self, conv):
        return len(self.render(conv)) // 3

    def generate(self, convs, max_tokens):
        out = []
        for c in convs:
            n = self.count_tokens(c)
            out.append(GenOut("", n, 0, "too_long") if n + max_tokens > self.max_model_len else GenOut(self.fn(c), n, 1, "stop"))
        return out


def make_llm(cfg: dict, mock_fn: Callable | None = None) -> BaseLLM:
    """cfg: backend (auto|vllm|hf|mock), model, revision, dtype, max_model_len, gpu_memory_utilization, quantization,
    load_in_4bit, enable_thinking, seed."""
    backend = cfg.get("backend", "auto")
    seed_everything(int(cfg.get("seed", 0)))
    if backend == "mock":
        return MockLLM(mock_fn or (lambda conv: "{\"done\": true}"), **cfg)
    if backend in ("auto", "vllm"):
        try:
            return VLLM(**{k: v for k, v in cfg.items() if k != "backend"})
        except Exception as e:  # not installed, or no kernel for this GPU (e.g. T4): fall back to transformers
            if backend == "vllm":
                raise
            print(f"[llm] vLLM unavailable ({type(e).__name__}: {e}); using transformers", flush=True)
    return HFLLM(**{k: v for k, v in cfg.items() if k != "backend"})

In [ ]:
%%writefile /kaggle/working/lib/dmlib/agent.py
"""DataMate code-agent: one generated solution per question, at most one repair after an execution error, abstain
when the final attempt still fails. Two architectures share everything except the shop context:

  raw    (lane C): README + the first raw rows of every file/sheet; the code reads the files itself with pandas.
  helper (lane D): profiler card of cleaned tables T[...] + helper library (periods P, helpers H, conversions CONV).
"""
from __future__ import annotations

import re
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd

from .executor import run_code
from .loader import read_csv_grid, read_excel_grids
from .rulebook import Rulebook

SYSTEM_COMMON = """Bạn là chuyên gia phân tích dữ liệu bán hàng của hộ kinh doanh Việt Nam. Hãy viết MỘT khối ```python``` duy nhất tính đáp án cho câu hỏi.

Quy tắc bắt buộc:
- Hôm nay là biến AS_OF (chuỗi 'YYYY-MM-DD'). Bản ghi có ngày sau hôm nay coi như chưa xảy ra (riêng trạng thái huỷ/trả hàng ghi trên đơn vẫn lấy).
- Áp dụng đúng RULEBOOK bên dưới; câu hỏi nói khác thì theo câu hỏi.
- Kết thúc code bằng đúng một lệnh:
  submit(type, value)  với type là number | percent | string | date | boolean | list | table
  unanswerable("lý do")  khi shop không ghi loại thông tin cần, hoặc kỳ hỏi nằm HOÀN TOÀN ngoài khoảng ngày có dữ liệu (kỳ chỉ chồng một phần thì vẫn tính trên phần có dữ liệu)
  abstain()  khi không đủ chắc chắn (sai bị trừ điểm, bỏ trống được 0 điểm).
- number: tiền là số nguyên VND (dùng submit('number', x, money=True) để làm tròn về đồng); số đếm là số nguyên; số khác làm tròn 2 chữ số.
- percent: là số phần trăm (12,4% -> 12.4), giảm thì âm, 2 chữ số thập phân.
- string: tên đúng như ghi trong dữ liệu (hoặc mã). date: 'YYYY-MM-DD' (câu hỏi theo tháng: 'YYYY-MM'). boolean: True/False.
- list: list Python có thứ tự (một phần tử vẫn là list); câu hỏi nhiều số thì theo đúng thứ tự câu hỏi nêu.
- table: {"columns": [tên cột...], "rows": [[...], ...]}; cột đầu là tên đối tượng, các cột sau là số đo theo thứ tự câu hỏi.
- Bằng nhau thì xếp theo mã tăng dần (không có mã thì theo tên).
- print() các số trung gian quan trọng (số dòng sau lọc, kỳ thời gian, tổng) để kiểm tra.
- Chỉ đọc file trong DATA_DIR và CALENDAR_PATH (lịch âm: cột date, lunar_day, lunar_month, lunar_year, lunar_leap). Không dùng mạng, không ghi/xoá file dữ liệu.
"""

SYSTEM_RAW = SYSTEM_COMMON + """
Dữ liệu là file THÔ trong thư mục DATA_DIR (xlsx/csv/sqlite). Bạn tự đọc bằng pandas (pd.read_excel(..., sheet_name=None, header=None), pd.read_csv, sqlite3), tự tìm dòng tiêu đề, tự xử lý: ô trống ở các dòng sau của cùng một đơn kế thừa dòng đầu; dòng trùng y hệt trong cùng một đơn chỉ tính một lần; tiền dạng '144.000đ', '750k', '1tr200', '1,45tr', '10,000.0000'; ngày dạng '1 thg 2 2025', '01/02/2025', ngày không ghi năm lấy năm của sheet; trạng thái chuẩn hoá về hoàn thành / huỷ / trả hàng / đang giao.
Biến có sẵn: DATA_DIR, CALENDAR_PATH, AS_OF, pd, np.
"""

SYSTEM_HELPER = SYSTEM_COMMON + """
Dữ liệu đã được đọc và làm sạch sẵn: T là dict tên bảng -> pandas DataFrame (mô tả bảng ở phần SHOP).
- Cột tiền/số đã là số; cột ngày là datetime64; cột số lượng kèm đơn vị có thêm cột '<tên>__unit'.
- Ô trống trong đơn đã được ffill (xem 'đã ffill trong đơn'); dòng trùng cùng đơn đã bỏ; cột _status là trạng thái chuẩn (hoàn thành / huỷ / trả hàng / đang giao) nếu có.
- Cột _file, _sheet, _row cho biết nguồn của dòng.
Thư viện có sẵn:
- P (dmlib.periods): P.this_week(AS_OF), P.last_week(AS_OF), P.last_n_days(AS_OF, n), P.this_month(AS_OF), P.same_days_prev_month(AS_OF), P.prev_month_full(AS_OF), P.month(m, year=None, as_of=AS_OF), P.quarter(q, year=None, as_of=AS_OF), P.year_to_date(AS_OF), P.day(d), P.yesterday(AS_OF), P.prev_period(p), P.clip_to_asof(p, AS_OF), P.tet_range(year, CAL), P.before_tet_range(year, CAL), P.shift_of(ts) -> 'sáng'|'chiều'|None, P.count_weekday(p, weekday_monday0), P.in_range(series, p) -> mask, P.overlap_status(p, (start, end)) -> 'outside'|'partial'|'inside'. Mỗi kỳ là tuple (date_bắt_đầu, date_kết_thúc) gồm cả hai đầu.
- H (dmlib.helpers): H.order_status(df, order_col) -> trạng thái mỗi đơn (dòng trả/huỷ đánh dấu cả đơn); H.revenue_lines(df, order_col) -> các dòng của đơn tính doanh thu; H.rank(df, value_col, name_col, code_col=None, k=None, ascending=False); H.base_factor(item, unit, CONV) -> hệ số quy về đơn vị cơ bản; H.money(x); H.pct(part, whole); H.pct_change(cur, prev).
- CONV: quy đổi đơn vị trong README {tên hàng: {đơn vị lớn: hệ số, '_base': đơn vị cơ bản}}; CAL: DataFrame lịch âm; PROFILE: hồ sơ shop (data_until, date_min, date_max, capabilities).
Biến có sẵn: T, P, H, CONV, CAL, PROFILE, AS_OF, DATA_DIR, CALENDAR_PATH, pd, np.
"""

PRELOAD_RAW = "import pandas as pd, numpy as np"
PRELOAD_HELPER = """import pandas as pd, numpy as np
from dmlib.profiler import load_cached_shop
from dmlib import helpers as H, periods as P
T, PROFILE = load_cached_shop(SHOP_CACHE)
CONV = PROFILE.get("conversions", {})
CAL = pd.read_csv(CALENDAR_PATH)"""

_CODE_RE = re.compile(r"```(?:python|py)?\s*\n(.*?)```", re.S)


def extract_code(text: str) -> str | None:
    """The last fenced block that ends the answer (submit/unanswerable/abstain), else the longest block."""
    blocks = [b.strip() for b in _CODE_RE.findall(text or "")]
    if blocks:
        final = [b for b in blocks if re.search(r"\b(submit|unanswerable|abstain)\(", b)]
        return final[-1] if final else max(blocks, key=len)
    t = (text or "").strip()
    if t.startswith("```"):
        t = t.strip("`").split("\n", 1)[-1]
    return t if re.search(r"\b(submit|unanswerable|abstain)\(", t) else None


def _cell(v, width=40) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).replace("\n", " ")
    return s if len(s) <= width else s[: width - 1] + "…"


def raw_shop_context(shop_dir: Path, head_rows: int = 8, max_chars: int = 9000) -> str:
    """README + the first raw rows (no header guessing) of every sheet / csv / sqlite table."""
    import sqlite3

    readme = (shop_dir / "README.md").read_text(encoding="utf-8") if (shop_dir / "README.md").exists() else ""
    parts = [readme.strip(), "", "## Các file (vài dòng đầu, chưa xử lý)"]
    for p in sorted(shop_dir.iterdir()):
        suf = p.suffix.lower()
        grids = {}
        if suf in (".xlsx", ".xlsm", ".xls"):
            for sh, g in read_excel_grids(p).items():
                grids[f"{p.name} | sheet '{sh}'"] = g
        elif suf in (".csv", ".tsv", ".txt"):
            grids[p.name] = read_csv_grid(p)
        elif suf in (".sqlite", ".db", ".sqlite3"):
            con = sqlite3.connect(f"file:{p}?mode=ro", uri=True)
            for (t,) in con.execute("select name from sqlite_master where type='table'"):
                n = con.execute(f'select count(*) from "{t}"').fetchone()[0]
                g = pd.read_sql_query(f'select * from "{t}" limit {head_rows}', con)
                g = pd.concat([pd.DataFrame([g.columns.tolist()]), pd.DataFrame(g.values)], ignore_index=True)
                grids[f"{p.name} | bảng sqlite '{t}' ({n} dòng)"] = g
            con.close()
        else:
            continue
        for title, g in grids.items():
            parts.append(f"\n### {title}  ({len(g)} dòng thô, {g.shape[1]} cột)")
            for _, row in g.head(head_rows).iterrows():
                parts.append(" | ".join(_cell(v) for v in row.tolist()))
    text = "\n".join(parts)
    return text if len(text) <= max_chars else text[:max_chars] + "\n…[cắt bớt]"


def helper_shop_context(shop_cache: Path, max_chars: int = 9000) -> str:
    text = (shop_cache / "card.md").read_text(encoding="utf-8")
    return text if len(text) <= max_chars else text[:max_chars] + "\n…[cắt bớt]"


@dataclass
class AgentConfig:
    mode: str = "helper"  # raw | helper
    max_new_tokens: int = 1536
    exec_timeout_s: float = 60
    exec_mem_mb: int = 6144
    exec_workers: int = 1
    repair: bool = True
    rules_k: int = 8
    context_chars: int = 9000


@dataclass
class Item:
    id: str
    business_id: str
    question: str
    as_of_date: str


@dataclass
class Outcome:
    record: dict
    status: str  # completed | retryable
    trace: dict = field(default_factory=dict)


class ShopContexts:
    """Per-shop prompt context and executor variables, computed once per shop."""

    def __init__(self, split_dir: Path, mode: str, cache_dir: Path | None, context_chars: int):
        self.split_dir, self.mode, self.cache_dir, self.context_chars = Path(split_dir), mode, cache_dir, context_chars
        self._ctx: dict[str, str] = {}

    def shop_dir(self, shop: str) -> Path:
        return self.split_dir / "data" / shop

    def text(self, shop: str) -> str:
        if shop not in self._ctx:
            if self.mode == "raw":
                self._ctx[shop] = raw_shop_context(self.shop_dir(shop), max_chars=self.context_chars)
            else:
                self._ctx[shop] = helper_shop_context(Path(self.cache_dir) / shop, max_chars=self.context_chars)
        return self._ctx[shop]

    def exec_context(self, item: Item) -> dict:
        ctx = {"DATA_DIR": str(self.shop_dir(item.business_id)), "CALENDAR_PATH": str(self.split_dir / "calendar.csv"), "AS_OF": item.as_of_date}
        if self.mode == "helper":
            ctx["SHOP_CACHE"] = str(Path(self.cache_dir) / item.business_id)
        return ctx

    def industry_text(self, shop: str) -> str:
        p = self.shop_dir(shop) / "README.md"
        return p.read_text(encoding="utf-8")[:1500] if p.exists() else ""


def build_messages(item: Item, shops: ShopContexts, rulebook: Rulebook, cfg: AgentConfig) -> list[dict]:
    system = SYSTEM_RAW if cfg.mode == "raw" else SYSTEM_HELPER
    rules = rulebook.select(item.question, industry_text=shops.industry_text(item.business_id), k=cfg.rules_k)
    user = (
        f"# RULEBOOK (trích)\n{rules}\n\n# SHOP {item.business_id}\n{shops.text(item.business_id)}\n\n"
        f"# CÂU HỎI\nHôm nay (AS_OF): {item.as_of_date}\nCâu hỏi: {item.question}\n\nViết code Python:"
    )
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def repair_messages(conv: list[dict], reply: str, error: str, stdout: str) -> list[dict]:
    msg = f"Code chạy lỗi:\n{(error or '')[-2500:]}\n"
    if stdout:
        msg += f"\nstdout:\n{stdout[-1500:]}\n"
    msg += "\nHãy sửa và trả về một khối ```python``` hoàn chỉnh (vẫn kết thúc bằng submit / unanswerable / abstain)."
    return conv + [{"role": "assistant", "content": reply}, {"role": "user", "content": msg}]


def _abstain(qid: str) -> dict:
    return {"id": qid, "type": "abstain"}


def solve_batch(items: list[Item], llm, rulebook: Rulebook, shops: ShopContexts, cfg: AgentConfig) -> list[Outcome]:
    t0 = time.time()
    preload = PRELOAD_RAW if cfg.mode == "raw" else PRELOAD_HELPER
    convs = [build_messages(it, shops, rulebook, cfg) for it in items]
    traces = [{"id": it.id, "shop": it.business_id, "question": it.question, "as_of": it.as_of_date, "mode": cfg.mode, "attempts": []} for it in items]
    try:
        gens = llm.generate(convs, cfg.max_new_tokens)
    except Exception as e:  # GPU/backend failure: keep pending, retry next session
        return [Outcome(_abstain(it.id), "retryable", {**tr, "error": f"llm: {type(e).__name__}: {e}"}) for it, tr in zip(items, traces)]

    def execute(i, text):
        code = extract_code(text)
        if code is None:
            return None, None
        return code, run_code(code, items[i].id, shops.exec_context(items[i]), timeout_s=cfg.exec_timeout_s, mem_mb=cfg.exec_mem_mb, preload=preload)

    with ThreadPoolExecutor(max_workers=max(1, cfg.exec_workers)) as pool:
        first = list(pool.map(lambda a: execute(*a), [(i, g.text) for i, g in enumerate(gens)]))
    results: list[dict | None] = [None] * len(items)
    need_repair = []
    for i, (g, (code, res)) in enumerate(zip(gens, first)):
        att = {"prompt_tokens": g.prompt_tokens, "output_tokens": g.output_tokens, "finish": g.finish, "reply": g.text, "code": code}
        if res is not None:
            att.update(ok=res.ok, error=res.error, stdout=res.stdout, exec_s=round(res.seconds, 2))
        else:
            att.update(ok=False, error="no code block in reply" if g.finish != "too_long" else "prompt too long")
        traces[i]["attempts"].append(att)
        if res is not None and res.ok:
            results[i] = res.record
        elif cfg.repair and g.finish != "too_long":
            need_repair.append(i)
    if need_repair:
        rconvs = [repair_messages(convs[i], gens[i].text, traces[i]["attempts"][0]["error"], traces[i]["attempts"][0].get("stdout", "")) for i in need_repair]
        try:
            rgens = llm.generate(rconvs, cfg.max_new_tokens)
        except Exception as e:
            rgens = None
            for i in need_repair:
                traces[i]["attempts"].append({"ok": False, "error": f"llm: {type(e).__name__}: {e}"})
        if rgens is not None:
            with ThreadPoolExecutor(max_workers=max(1, cfg.exec_workers)) as pool:
                second = list(pool.map(lambda a: execute(*a), [(i, g.text) for i, g in zip(need_repair, rgens)]))
            for i, g, (code, res) in zip(need_repair, rgens, second):
                att = {"prompt_tokens": g.prompt_tokens, "output_tokens": g.output_tokens, "finish": g.finish, "reply": g.text, "code": code}
                att.update(ok=bool(res and res.ok), error=(res.error if res else "no code block in reply"), stdout=(res.stdout if res else ""))
                traces[i]["attempts"].append(att)
                if res is not None and res.ok:
                    results[i] = res.record
    out = []
    per = (time.time() - t0) / max(len(items), 1)
    for it, rec, tr in zip(items, results, traces):
        final = rec if rec is not None else _abstain(it.id)
        tr["final"] = final
        tr["abstain_reason"] = None if rec is not None else (tr["attempts"][-1].get("error") or "failed")
        tr["seconds_per_item_batch"] = round(per, 2)
        out.append(Outcome(final, "completed", tr))
    return out

In [ ]:
%%writefile /kaggle/working/lib/dmlib/split.py
"""Fixed shop-level manifest for train: train-fit / dev-tune / final-holdout (no shop in two parts)."""
from __future__ import annotations

import random
from collections import Counter


def make_manifest(questions: list[dict], seed: int = 2026, dev_shops: int = 6, final_shops: int = 6) -> dict:
    shops = sorted({q["business_id"] for q in questions})
    rng = random.Random(seed)
    rng.shuffle(shops)
    final, dev, fit = shops[:final_shops], shops[final_shops : final_shops + dev_shops], shops[final_shops + dev_shops :]
    part = {s: "final_holdout" for s in final} | {s: "dev_tune" for s in dev} | {s: "train_fit" for s in fit}
    ids = {"train_fit": [], "dev_tune": [], "final_holdout": []}
    for q in questions:
        ids[part[q["business_id"]]].append(q["id"])
    return {
        "seed": seed,
        "shops": {"train_fit": sorted(fit), "dev_tune": sorted(dev), "final_holdout": sorted(final)},
        "ids": ids,
        "counts": {k: len(v) for k, v in ids.items()},
        "questions_per_shop": dict(Counter(q["business_id"] for q in questions)),
    }

In [ ]:
%%writefile /kaggle/working/lib/dmlib/run.py
"""DataMate worker: `python -m dmlib.run --config cfg.json --shard I` answers the questions of shard I (stable hash of
the id) and appends one durable record per question; rerunning the same config resumes where it stopped.

`launch(cfg)` starts one worker per GPU (CUDA_VISIBLE_DEVICES=i) and waits; merge.py builds the submission.
"""
from __future__ import annotations

import argparse
import json
import os
import subprocess
import sys
import time
from pathlib import Path

from .agent import AgentConfig, Item, ShopContexts, solve_batch
from .common import DurableJsonl, Deadline, atomic_write_json, config_hash, env_info, find_split_dir, load_jsonl, set_thread_env, shard_of
from .rulebook import Rulebook
from .split import make_manifest

RUN_IGNORE = ("shard_index", "num_shards", "out_dir", "resume", "wallclock_s", "parent_run", "notes", "input_roots", "split_dir", "cache_dir")


def resolve_split_dir(cfg: dict) -> Path:
    if cfg.get("split_dir"):
        return Path(cfg["split_dir"])
    return find_split_dir(cfg["split"], cfg.get("input_roots", ["/kaggle/input"]), marker="questions.jsonl")


def select_items(cfg: dict, split_dir: Path) -> list[Item]:
    qs = load_jsonl(split_dir / "questions.jsonl")
    subset = cfg.get("subset", "all")
    if subset != "all":
        if isinstance(subset, list):
            keep = set(subset)
        else:
            keep = set(make_manifest(qs, seed=cfg.get("manifest_seed", 2026))["ids"][subset])
        qs = [q for q in qs if q["id"] in keep]
    if cfg.get("shops"):
        qs = [q for q in qs if q["business_id"] in set(cfg["shops"])]
    if cfg.get("limit"):
        qs = qs[: int(cfg["limit"])]
    return [Item(q["id"], q["business_id"], q["question"], q["as_of_date"]) for q in qs]


def run_hash(cfg: dict) -> str:
    return config_hash(cfg, ignore=RUN_IGNORE)


def _gpu_mem() -> list[str]:
    try:
        return subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader"],
                              capture_output=True, text=True, timeout=20).stdout.strip().splitlines()
    except Exception:
        return []


def work(cfg: dict, shard_index: int, llm=None) -> dict:
    set_thread_env(int(cfg.get("cpu_threads", 1)))
    deadline = Deadline(float(cfg.get("wallclock_s", 3.5 * 3600)))
    split_dir = resolve_split_dir(cfg)
    out = Path(cfg["out_dir"])
    out.mkdir(parents=True, exist_ok=True)
    rh = run_hash(cfg)
    n_shards = int(cfg.get("num_shards", 1))
    items = [it for it in select_items(cfg, split_dir) if shard_of(it.id, n_shards) == shard_index]
    preds = DurableJsonl(out / f"preds.shard{shard_index}.jsonl", rh)
    traces = DurableJsonl(out / f"traces.shard{shard_index}.jsonl", rh)
    done = preds.done_ids() if cfg.get("resume", True) else set()
    todo = sorted([it for it in items if it.id not in done], key=lambda it: (it.business_id, it.id))
    acfg = AgentConfig(mode=cfg["mode"], **cfg.get("agent", {}))
    shops = ShopContexts(split_dir, acfg.mode, Path(cfg["cache_dir"]) / cfg["split"] if acfg.mode == "helper" else None, acfg.context_chars)
    rulebook = Rulebook((split_dir / "rulebook.md").read_text(encoding="utf-8"))
    if acfg.mode == "helper":
        missing = sorted({it.business_id for it in todo if not (shops.cache_dir / it.business_id / "COMPLETE").exists()})
        if missing:
            raise FileNotFoundError(f"profiler cache missing for shops {missing} under {shops.cache_dir}")
    t_load = time.time()
    if llm is None:
        from .llm import make_llm

        llm = make_llm(cfg["llm"])
    load_s = time.time() - t_load
    bs = int(cfg.get("batch_size", 16))
    n_done = 0
    t_run = time.time()
    stopped_early = False
    for b in range(0, len(todo), bs):
        if deadline.expired():
            stopped_early = True
            break
        batch = todo[b : b + bs]
        for o in solve_batch(batch, llm, rulebook, shops, acfg):
            traces.append({"id": o.record["id"], "status": o.status, **o.trace})
            preds.append({"id": o.record["id"], "status": o.status, "record": o.record})
            n_done += o.status == "completed"
        print(f"[shard {shard_index}] {b + len(batch)}/{len(todo)} elapsed {deadline.elapsed():.0f}s", flush=True)
    preds.close()
    traces.close()
    summary = {
        "shard": shard_index,
        "run_hash": rh,
        "items_in_shard": len(items),
        "already_done": len(done),
        "completed_now": n_done,
        "remaining": len(todo) - n_done,
        "stopped_early_by_wallclock": stopped_early,
        "model_load_s": round(load_s, 1),
        "run_s": round(time.time() - t_run, 1),
        "s_per_item": round((time.time() - t_run) / max(n_done, 1), 2),
        "gpu_mem_end": _gpu_mem(),
        "env": env_info(),
    }
    atomic_write_json(out / f"summary.shard{shard_index}.json", summary)
    return summary


def launch(cfg: dict, cfg_path: str, gpus: list[int] | None = None, lib_root: str | None = None, python: str | None = None) -> list[int]:
    """One worker process per shard; shard i sees GPU gpus[i]. Returns exit codes; logs in out_dir/worker<i>.log."""
    out = Path(cfg["out_dir"])
    out.mkdir(parents=True, exist_ok=True)
    n = int(cfg.get("num_shards", 1))
    gpus = gpus if gpus is not None else list(range(n))
    procs = []
    for i in range(n):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpus[i % max(len(gpus), 1)]) if gpus else "", PYTHONUNBUFFERED="1")
        if lib_root:
            env["PYTHONPATH"] = lib_root + os.pathsep + env.get("PYTHONPATH", "")
        log = open(out / f"worker{i}.log", "a", encoding="utf-8")
        procs.append((subprocess.Popen([python or sys.executable, "-m", "dmlib.run", "--config", cfg_path, "--shard", str(i)], env=env, stdout=log, stderr=subprocess.STDOUT), log))
    codes = []
    for p, log in procs:
        codes.append(p.wait())
        log.close()
    return codes


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--shard", type=int, default=0)
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    print(json.dumps(work(cfg, a.shard), ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/lib/dmlib/merge.py
"""Merge worker shards into submission.jsonl, validate every line, and score when the split has answers."""
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path

from .answer import penalized_accuracy, validate_record
from .common import atomic_write_json, atomic_write_text, dumps, load_jsonl, read_jsonl_tolerant


def collect(out_dir: Path) -> dict[str, dict]:
    """Latest completed record per id across shards (header lines skipped)."""
    recs: dict[str, dict] = {}
    for p in sorted(Path(out_dir).glob("preds.shard*.jsonl")):
        rows, _ = read_jsonl_tolerant(p)
        for r in rows[1:]:
            if r.get("status") == "completed":
                recs[r["id"]] = r["record"]
    return recs


def merge(out_dir, split_dir, finalize: bool = False, submission_name: str = "submission.jsonl", ids: list[str] | None = None) -> dict:
    """finalize=True writes abstain for ids still pending (accepting that coverage); otherwise only reports them."""
    out_dir, split_dir = Path(out_dir), Path(split_dir)
    questions = load_jsonl(split_dir / "questions.jsonl")
    order = ids if ids is not None else [q["id"] for q in questions]
    shop = {q["id"]: q["business_id"] for q in questions}
    recs = collect(out_dir)
    pending = [i for i in order if i not in recs]
    bad = {}
    lines = []
    for qid in order:
        rec = recs.get(qid)
        if rec is None:
            if not finalize:
                continue
            rec = {"id": qid, "type": "abstain"}
        err = validate_record(rec)
        if err:
            bad[qid] = err
            rec = {"id": qid, "type": "abstain"}
        lines.append(dumps(rec))
    report = {
        "n_questions": len(order),
        "n_written": len(lines),
        "pending": len(pending),
        "pending_ids_head": pending[:20],
        "finalized": finalize,
        "invalid_replaced_by_abstain": bad,
        "type_counts": dict(Counter(recs[i]["type"] for i in order if i in recs)),
    }
    if not pending or finalize:
        atomic_write_text(out_dir / submission_name, "\n".join(lines) + "\n")
        report["submission"] = str(out_dir / submission_name)
    ans_path = split_dir / "answers.jsonl"
    if ans_path.exists():
        golds = {g["id"]: g for g in load_jsonl(ans_path) if g["id"] in set(order)}
        preds = {i: recs[i] for i in order if i in recs}
        pa = penalized_accuracy(preds, golds)
        per_id = pa.pop("per_id")
        report["score"] = pa
        by_shop, by_type = defaultdict(list), defaultdict(list)
        for qid, s in per_id.items():
            by_shop[shop[qid]].append(s)
            by_type[golds[qid]["type"]].append(s)
        summ = lambda v: {"n": len(v), "pa": round(sum(v) / len(v), 4), "correct": v.count(1), "wrong": v.count(-1), "abstain": v.count(0)}
        report["by_shop"] = {k: summ(v) for k, v in sorted(by_shop.items())}
        report["by_gold_type"] = {k: summ(v) for k, v in sorted(by_type.items())}
        atomic_write_text(out_dir / "per_id_scores.jsonl", "\n".join(dumps({"id": k, "score": v, "shop": shop[k], "gold_type": golds[k]["type"]}) for k, v in per_id.items()) + "\n")
    atomic_write_json(out_dir / "merge_report.json", report)
    return report

## Cài đặt và kiểm tra môi trường

In [ ]:
# Thư viện cài thêm: ưu tiên wheel từ output notebook `zalo2026-wheels` (Add Input), không có / lỗi thì cài từ mạng
WHEEL_DIRS = sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/wheels/*.whl", recursive=True)})
print("wheel dirs:", WHEEL_DIRS or "none -> install from internet")

def pip_install(pkgs: str) -> int:
    if WHEEL_DIRS:
        links = " ".join(f"--find-links {d}" for d in WHEEL_DIRS)
        if sh(f"{sys.executable} -m pip install -q --no-index {links} {pkgs}") == 0:
            print("installed from wheels:", pkgs)
            return 0
        print("wheel install failed -> internet:", pkgs)
    return sh(f"{sys.executable} -m pip install -q {pkgs}")

pip_install("python-calamine")
# vLLM goes into its own venv: `pip install vllm` into the Kaggle env replaces torch (2.13+cu130 on the 2026-10 image)
# and breaks torchaudio -> transformers ASR. The base env stays untouched for ASR and the transformers fallback.
LLM_PYTHON = sys.executable
if CONFIG.get("llm", {}).get("backend", "auto") in ("auto", "vllm") and CONFIG.get("install_vllm", True):
    VENV = Path("/tmp/venv_vllm")  # not under /kaggle/working: it would be saved as output (GBs)
    pip_install("uv")
    sh(f"{sys.executable} -m uv venv -q {VENV} --python {sys.executable}")
    vpy = VENV / "bin" / "python"
    pkgs = f"{CONFIG.get('vllm_spec', 'vllm==0.32.0 wrapt')} pandas pyarrow openpyxl python-calamine"
    rc = 1
    if WHEEL_DIRS:
        links = " ".join(f"--find-links {d}" for d in WHEEL_DIRS)
        rc = sh(f"{sys.executable} -m uv pip install -q --python {vpy} --no-index {links} {pkgs}")
    if rc != 0:
        rc = sh(f"{sys.executable} -m uv pip install -q --python {vpy} {pkgs}")
    check = "import vllm, torch; assert torch.cuda.is_available(); print('vllm', vllm.__version__, 'torch', torch.__version__)"
    if rc == 0 and sh(f'{vpy} -c "{check}"') == 0:
        LLM_PYTHON = str(vpy)
print("LLM worker python:", LLM_PYTHON, "(transformers fallback)" if LLM_PYTHON == sys.executable else "(vLLM venv)")

In [ ]:
# Dataset: dùng bản đã Add Input trong /kaggle/input; nếu chưa có thì tự tải bằng kagglehub
# (https://www.kaggle.com/datasets/thanhkhil/zaloai-datamate-dataset)
DATASET = "thanhkhil/zaloai-datamate-dataset"
if not glob.glob(f"/kaggle/input/**/{CONFIG['split']}/questions.jsonl", recursive=True):
    pip_install("kagglehub")
    import kagglehub
    DATA_PATH = kagglehub.dataset_download(DATASET)
    print("dataset downloaded to", DATA_PATH)
    CONFIG["input_roots"] = [str(DATA_PATH)] + CONFIG["input_roots"]
else:
    print("dataset found under /kaggle/input")

In [ ]:
import importlib
for m in list(sys.modules):
    if m == "dmlib" or m.startswith("dmlib."):
        del sys.modules[m]
from dmlib.common import env_info, find_split_dir, atomic_write_json
info = env_info(); print(json.dumps(info, indent=1, ensure_ascii=False))
N_GPUS = len(info.get("gpus", []))
SPLIT_DIR = find_split_dir(CONFIG["split"], CONFIG["input_roots"], marker="questions.jsonl")
print("split dir:", SPLIT_DIR, "| shops:", len(list((SPLIT_DIR / "data").iterdir())), "| has answers:", (SPLIT_DIR / "answers.jsonl").exists())
atomic_write_json(OUT / "environment.json", info)

In [ ]:
# Helper mode needs the profiler cache: use an attached output of datamate_profiler if it has this split, else build it here (CPU, a few minutes)
from dmlib.profiler import profile_shop
found = sorted({str(Path(p).parent.parent.parent) for p in glob.glob(f"/kaggle/input/**/{CONFIG['split']}/*/COMPLETE", recursive=True) if (Path(p).parent / "card.md").exists()})
shops = sorted(p.name for p in (SPLIT_DIR / "data").iterdir() if p.is_dir())
if found and all((Path(found[0]) / CONFIG["split"] / s / "COMPLETE").exists() for s in shops):
    CONFIG["cache_dir"] = found[0]; print("using attached profiler cache:", found[0])
else:
    rb_sha = __import__("hashlib").sha256((SPLIT_DIR / "rulebook.md").read_bytes()).hexdigest()[:12]
    for s in shops:
        t = time.time(); p = profile_shop(SPLIT_DIR / "data" / s, Path(CONFIG["cache_dir"]) / CONFIG["split"], split=CONFIG["split"], rulebook_sha=rb_sha)
        print(f"{s}: {len(p['tables'])} tables, {time.time() - t:.1f}s, errors={p['errors']}")

In [ ]:
# Resume: attach the output of an earlier version of this notebook as input; its shard files are copied back here
# (same config only: a different config hash is refused by the worker).
for d in CONFIG.get("resume_from_dirs", []):
    for p in glob.glob(f"{d}/**/preds.shard*.jsonl", recursive=True) + glob.glob(f"{d}/**/traces.shard*.jsonl", recursive=True):
        dst = OUT / Path(p).name
        if not dst.exists():
            shutil.copy(p, dst); print("resume <-", p)

## Chạy (một worker / GPU, shard theo id ổn định, ghi bền từng câu)

In [ ]:
from dmlib.run import launch, select_items
from dmlib.merge import merge
cfg = dict(CONFIG, split_dir=str(SPLIT_DIR), num_shards=max(N_GPUS, 1))
cfg_path = OUT / "config.json"; atomic_write_json(cfg_path, cfg)
t0 = time.time()
codes = launch(cfg, str(cfg_path), gpus=list(range(N_GPUS)), lib_root=str(LIB), python=LLM_PYTHON)
print("exit codes:", codes, f"{(time.time() - t0) / 60:.1f} min")
for log in sorted(OUT.glob("worker*.log")):
    print("=====", log.name); print(log.read_text(encoding="utf-8")[-2500:])

In [ ]:
ids = [it.id for it in select_items(cfg, SPLIT_DIR)]
rep = merge(cfg["out_dir"], SPLIT_DIR, finalize=CONFIG["finalize"], ids=ids)
print(json.dumps({k: v for k, v in rep.items() if k not in ("by_shop",)}, ensure_ascii=False, indent=1))
if rep.get("by_shop"):
    for s, v in rep["by_shop"].items(): print(s, v)
answered = rep["n_questions"] - rep["pending"]
if rep.get("submission") and CONFIG["split"] == "public_test":
    if answered == 0:
        # every line would be an abstain filled in for a crash: 0 points and a wasted daily submission
        print("NOT exporting /kaggle/working/submission.jsonl: no question was processed (see worker logs above)")
    else:
        shutil.copy(rep["submission"], WORK / "submission.jsonl")
        print(f"-> /kaggle/working/submission.jsonl ({answered}/{rep['n_questions']} processed, rest abstain)")
for p in sorted(OUT.glob("summary.shard*.json")):
    print(p.name, json.loads(p.read_text())["s_per_item"], "s/question")

In [ ]:
# A few traces to eyeball (question, code, error, final answer)
tr = [json.loads(l) for p in sorted(OUT.glob("traces.shard*.jsonl")) for l in open(p, encoding="utf-8").read().splitlines()[1:]]
for t in tr[:3]:
    print("Q:", t["question"], "| as_of", t["as_of"]); a = t["attempts"][-1]
    print(a.get("code")); print("error:", (a.get("error") or "")[-500:]); print("final:", t["final"]); print("-" * 80)